# 05 — Train: head iden + head OS + cascade hostname Data_AKien

Một notebook train head iden/OS và cascade hostname thử nghiệm, ghi **một** run khi chạy toàn bộ:

```
Models/<ts>/
├── train/            notebook này
│   ├── iden.joblib   phần A — encoder + 3 rừng make / type / model (12 thiết bị FIELD, CIC-2022 làm nền __unknown__)
│   ├── os.joblib     phần B — head OS: encoder + rừng + chữ ký PRL
│   ├── hostname_cascade.joblib   phần C — luật A → expert B → __unknown__
│   ├── hostname_*.csv/json      báo cáo so sánh A/B/cascade, thresholds, gate, trace
│   └── meta.json     số liệu train; thêm provenance hostname khi chạy phần C
├── iden/             09: sdc_iden.onnx + báo cáo
└── release/          12: sdc_os.onnx + manifest.json; 13 thêm sdc_os.tflite
```

Bước sau: **09** đóng gói `iden.joblib` thành `sdc_iden.onnx`; **12** ghép `os.joblib` của **cùng run** vào đó thành
bản release; **13** xuất TFLite. Muốn sửa cách train thì sửa ở đây; hàm dùng chung (encoder, `make_model`, luật
OS, `cascade`) ở mục **Thư viện** đầu notebook. Phần C dùng data hostname trong `Data_AKien`; có thể chạy riêng.
Cascade hostname đi cùng pipeline 09 → 12 → 13; đặt cùng RUN ở cả ba notebook.

In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


## Thư viện

Hàm và hằng dùng chung, chép **nguyên văn** (không còn notebook `00_common`). Notebook khác có bản sao của cùng hàm — sửa ở đây thì sửa cả các bản kia cho khớp: bảng *Hàm có bản sao ở đâu* trong `README.md`. Phần riêng của notebook này bắt đầu sau mục này.

In [2]:
# --- 1. Nền: đường dẫn, lược đồ, mã hoá ---
# Xác định gốc dự án từ thư mục làm việc của kernel Jupyter.
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ bên trong dự án SDC")
print("Gốc dự án:", ROOT)

Gốc dự án: D:\01.AI_Security\02.SDC


In [3]:
# --- Đường dẫn dự án ---
from pathlib import Path


DATA = ROOT / "Data"
MODELS = ROOT / "Models"
SESSIONS_PATH = DATA / "sessions_verified.parquet"

In [4]:
# --- Lược đồ dữ liệu — head, nhóm feature, chia tập ---
import pandas as pd


MISSING = "<missing>"

META_COLS = ["canonical_device", "scenario", "capture_file", "date", "mac", "n_sources"]

# Ba head, đúng thứ tự output của sản phẩm: hãng / loại / model sản phẩm.
#
# Head thứ ba từng tên là `family` (contract ≤ 2.1). Đổi thành `model` cho khớp ô MODEL
# trong spec, nhưng chỉ là ĐỔI TÊN — nội dung nhãn không đổi, và độ mịn của nó vẫn là
# "model sản phẩm ở mức phân biệt được bằng telemetry": `Amazon Echo Dot` là một model,
# còn `Tuya Plug` vẫn gộp 13 thiết bị ODM không tách được bằng DHCP/DNS/TLS. Đổi tên
# không làm chúng tách ra được; xem `Data/device_labels_verified.csv` cột review_note.
MODEL_HEAD = "model"
LEGACY_MODEL_HEAD = "family"
HEAD_ALIASES = {LEGACY_MODEL_HEAD: MODEL_HEAD}
LABEL_COLS = ["make", "type", MODEL_HEAD]
SOURCES = ["dhcp", "dns", "mdns", "tls"]

# Cột dạng chuỗi token -> TF-IDF. Phần còn lại dạng chuỗi -> categorical nguyên giá trị.
TEXT_COLS = ["dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens"]

# Feature biến mất khi DNS được mã hoá (DoH/DoT) và SNI bị ẩn (ECH). Đây là lý do thật
# để đo bản không có chúng — không phải vì sợ leak: lúc chạy thật router vẫn đọc được
# DNS query và SNI, nên chúng là bằng chứng hợp lệ.
ENCRYPTED_RISK = ["dns_tokens", "tls_sni_tokens"]

# Field driver KHÔNG cung cấp — không feature nào được suy ra từ chúng
FORBIDDEN = ["src_ip", "dst_ip", "dst_port", "ts", "iat", "pktsize", "bytes", "rate"]


# Cot co-mat cua tung nguon. Suy tu `SOURCES` de danh sach nguon chi co MOT ban.
SOURCE_FLAGS = tuple(f"has_{source}" for source in SOURCES)

RARE_THRESHOLD = 10      # lớp dưới ngưỡng này không học được, tách riêng khi report


def feature_sets(df):
    """Các nhóm feature, suy từ chính các cột của dataframe."""
    feats = [c for c in df.columns if c not in META_COLS + LABEL_COLS]
    text = [c for c in TEXT_COLS if c in feats]
    cat = [c for c in feats if c not in text and not pd.api.types.is_numeric_dtype(df[c])]
    num = [c for c in feats if c not in text and c not in cat]
    return {
        "all": feats,
        "text": text,
        "cat": cat,
        "num": num,
        "source": {s: [c for c in feats if c.startswith(s)] for s in SOURCES},
        "no_content": [c for c in feats if c not in ENCRYPTED_RISK],
    }


def assert_serveable(cols):
    """Fail nếu có feature vượt quá telemetry driver cung cấp."""
    bad = [c for c in cols if any(f in c for f in FORBIDDEN)]
    assert not bad, f"Feature không phục vụ được lúc chạy thật: {bad}"
    orphan = [c for c in cols if not c.startswith(("dhcp_", "dns_", "mdns_", "tls_", "has_"))]
    assert not orphan, f"Feature không truy được về nguồn nào: {orphan}"
    return True


def normalize_heads(df):
    """Đổi tên cột nhãn cũ sang tên head hiện hành (`family` -> `model`).

    Làm ở đây, một chỗ, thay vì ghi lại parquet: notebook 03_build_dataset và các file dữ liệu đã sinh
    ra từ trước vẫn dùng được, mà phần còn lại của code chỉ biết đúng một tên. Cột mới
    đã có sẵn thì không đụng vào — dữ liệu mới luôn thắng.
    """
    rename = {old: new for old, new in HEAD_ALIASES.items()
              if old in df.columns and new not in df.columns}
    # Có cả hai cột: cột mới là nhãn đang dùng, cột cũ là tàn dư của file dựng trước lần
    # đổi tên. Bỏ hẳn cột cũ — để lại thì nó không phải nhãn mà cũng không phải feature,
    # và `assert_serveable` sẽ bắt nó như một feature không truy được về nguồn nào.
    stale = [old for old, new in HEAD_ALIASES.items()
             if old in df.columns and new in df.columns]
    if rename or stale:
        df = df.rename(columns=rename).drop(columns=stale)
    return df


def load_sessions(path=SESSIONS_PATH):
    """Nạp feature table mức session kèm nhóm feature đã kiểm tra."""
    assert path.exists(), f"Không tìm thấy {path} — chạy 03_build_dataset.ipynb trước"
    df = normalize_heads(pd.read_parquet(path))
    fs = feature_sets(df)
    assert_serveable(fs["all"])
    return df, fs

In [5]:
# --- Bộ phân lớp L2 ---
from sklearn.ensemble import RandomForestClassifier

# Số cây của RandomForest. Một định nghĩa cho cả 03 (train), 04 (hiệu chỉnh ngưỡng) và
# 05 (xuất ONNX) — ba chỗ này BẮT BUỘC dùng chung một giá trị, vì ngưỡng abstain hiệu
# chỉnh cho model 250 cây không đúng với model 25 cây.
#
# Đo được (CV 5 fold theo ngày): 25 / 100 / 250 cây cho macro-F1 như nhau trong biên độ
# nhiễu — `type` lần lượt 0.8442 / 0.8443 / 0.8440, và 50 cây còn ra 0.8331, tức không
# đơn điệu. Kích thước thì khác hẳn: 1.54 / 6.07 / 15.80 MB.
#
# ⚠️ Số cây ảnh hưởng tới ĐỘ PHÂN GIẢI của confidence mạnh hơn tới accuracy: 25 cây chỉ
# sinh được 26 mức xác suất (bội của 1/25), nên đường coverage–accuracy thành bậc thang
# thô và ngưỡng chọn được kém tinh. Với kiến trúc này, confidence là thứ quyết định phát
# hiện thiết bị lạ, nên đổi số cây thì phải đo lại `ood_abstain` chứ đừng chỉ nhìn F1.
N_ESTIMATORS = 250

RNG = 42


def make_model() -> RandomForestClassifier:
    """Bo phan lop chuan cua du an. Train va danh gia BAT BUOC goi cung ham nay."""
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RNG,
    )

In [6]:
# --- Bộ mã hoá feature (TF-IDF + ordinal) ---
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OrdinalEncoder


# Số chiều TF-IDF mỗi cột text. Cộng lại ra kích thước input tensor nên phải cố định
# và ghi vào model contract.
TFIDF_MAX_FEATURES = {"dns_tokens": 200, "mdns_tokens": 100,
                      "tls_sni_tokens": 150, "dhcp_vci": 100}

# --- Encode -------------------------------------------------------------------------
# Bộ mã hoá là một `ColumnTransformer` của sklearn, không phải code tự ghép. Lý do không
# phải vì gọn hơn, mà vì **xuất được sang ONNX**: skl2onnx chuyển được cả cụm
# ColumnTransformer + TF-IDF + RandomForest thành MỘT graph nhận chuỗi thô, nên user space
# không phải cài lại TF-IDF — nguồn train/serve skew nguy hiểm nhất của pipeline này.
#
# Hai quy ước đổi so với contract 1.x, đều vì ràng buộc của skl2onnx:
#   - `dhcp_vci` dùng analyzer 'word' tách theo ':' '-' '.' '_' thay vì char_wb 3-5gram.
#     skl2onnx chỉ chuyển được analyzer='word'. Đo được (CV 5 fold theo ngày): cách mới
#     KHÔNG tệ hơn — macro-F1 `type` 0.8331 -> 0.8440, make/family không đổi. Tách
#     'dhcpcd-6.8.2:Linux-4.4.22+:armv7l:MT8167B' theo dấu phân cách cho ra token có
#     nghĩa (dhcpcd, Linux, armv7l, MT8167B), hoá ra hợp với trường này hơn char n-gram.
#   - Chỉ số OOV là -1, không phải len(vocab). Đây là quy ước của `OrdinalEncoder`.
#
# Cả hai đổi đều phá contract cũ nên `contract_version` phải là 2.x.

VCI_TOKEN_PATTERN = r"[^:\-_. ]+"   # dhcp_vci: tách theo dấu phân cách trong chính chuỗi
TOKEN_PATTERN = r"[^ ]+"            # còn lại: bước 02 đã tách token sẵn, chỉ cắt khoảng trắng
OOV_INDEX = -1                      # giá trị categorical chưa từng thấy


def _make_vectorizer(col):
    pattern = VCI_TOKEN_PATTERN if col == "dhcp_vci" else TOKEN_PATTERN
    return TfidfVectorizer(analyzer="word", token_pattern=pattern,
                           max_features=TFIDF_MAX_FEATURES[col], min_df=2)


def build_encoder(num, cat, text):
    """ColumnTransformer chuẩn của project. Thứ tự khối: numeric, categorical, rồi text."""
    steps = []
    if num:
        steps.append(("num", "passthrough", list(num)))
    if cat:
        steps.append(("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                            unknown_value=OOV_INDEX,
                                            encoded_missing_value=OOV_INDEX), list(cat)))
    for c in text:
        steps.append((f"t_{c}", _make_vectorizer(c), c))   # chuỗi, không phải list
    return ColumnTransformer(steps, sparse_threshold=0)


def prepare_frame(df, encoder):
    """Ép kiểu đúng như lúc fit: numeric float32, categorical/text chuỗi.

    ColumnTransformer khớp cột theo TÊN nên thứ tự cột đầu vào không quan trọng, nhưng
    kiểu thì có: TF-IDF nhận object/str, để lọt NaN vào là vỡ.
    """
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    out = df[num + cat + text].copy()
    for c in num:
        out[c] = out[c].astype(np.float32)
    for c in cat + text:
        out[c] = out[c].astype(str)
    return out


def fit_encoder(df, cols, min_count=1):
    """Trả dict mô tả toàn bộ phép encode — chính là phần `vocab` của model contract."""
    fs = feature_sets(df)
    num = [c for c in fs["num"] if c in cols]
    cat = [c for c in fs["cat"] if c in cols]
    text = [c for c in fs["text"] if c in cols]
    encoder = {"num_cols": num, "cat_cols": cat, "text_cols": text,
               "ct": build_encoder(num, cat, text)}
    encoder["ct"].fit(prepare_frame(df, encoder))
    return encoder


def encoder_vectorizers(encoder):
    """TF-IDF đã fit, theo cột — 05 cần để ghi tham số vào contract."""
    named = encoder["ct"].named_transformers_
    return {c: named[f"t_{c}"] for c in encoder["text_cols"]}


def apply_encoder(df, encoder):
    """Trả (X, feature_names, cat_index). cat_index dùng cho HistGradientBoosting."""
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    X = encoder["ct"].transform(prepare_frame(df, encoder)).astype(np.float32)
    names = list(num) + list(cat)
    vecs = encoder_vectorizers(encoder)
    for c in text:
        names += [f"{c}::{t}" for t in vecs[c].get_feature_names_out()]
    cat_index = list(range(len(num), len(num) + len(cat)))
    return X, names, cat_index

In [7]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
import hashlib
import re
import struct
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


try:  # Packet parsing is not needed by record-only production deployments.
    from scapy.all import BOOTP, DHCP, DNS, Ether, IP, IPv6, PcapReader, Raw, TCP, UDP
except ImportError:  # pragma: no cover - exercised only in minimal production images
    BOOTP = DHCP = DNS = Ether = IP = IPv6 = PcapReader = Raw = TCP = UDP = None


MAX_TOKENS = 40
TOKEN_SPLIT = re.compile(r"[.\-]")

RAW_FIELDS = {
    "dhcp": ("opt55", "opt60"),
    "dns": ("qname",),
    "mdns": ("qname",),
    "tls": ("version", "ciphers", "alpn", "sni"),
}
# Cột cờ có-mặt nằm ở `sdc_schema` vì đó là lược đồ chung, không riêng của bước parse.
# Gác này bắt trường hợp hai nơi kể tên nguồn lệch nhau — im lặng thì `n_sources` sai.
assert tuple(RAW_FIELDS) == tuple(SOURCES), (
    f"RAW_FIELDS {tuple(RAW_FIELDS)} lệch SOURCES {tuple(SOURCES)}"
)


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
        return bool(missing) if isinstance(missing, (bool, np.bool_)) else False
    except (TypeError, ValueError):
        return False


def _text(value, *, lowercase=False):
    if _is_missing(value):
        return None
    if isinstance(value, bytes):
        value = value.decode(errors="replace")
    value = str(value).strip()
    if not value:
        return None
    return value.lower() if lowercase else value

In [8]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def is_grease(value):
    """RFC 8701 GREASE values have both bytes equal and a low nibble of 0xA."""
    try:
        value = int(value)
    except (TypeError, ValueError):
        return False
    return 0 <= value <= 0xFFFF and (value & 0x0F0F) == 0x0A0A


def normalize_cipher_suites(value):
    if _is_missing(value):
        return MISSING
    values = value if isinstance(value, (list, tuple)) else re.findall(r"\d+", str(value))
    normalized = []
    for item in values:
        try:
            number = int(item)
        except (TypeError, ValueError):
            continue
        if not is_grease(number):
            normalized.append(str(number))
    return ",".join(normalized) if normalized else MISSING


def normalize_tls_version(value):
    if _is_missing(value):
        return MISSING
    text = str(value).strip().lower()
    if text.startswith("0x"):
        text = text[2:]
    if re.fullmatch(r"\d{1,4}", text):
        return text.zfill(4)
    return MISSING

In [9]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def domain_tokens(names, max_tokens=MAX_TOKENS):
    """Domain names to unique lowercase tokens, preserving first-seen order."""
    tokens = []
    for name in names:
        if _is_missing(name):
            continue
        normalized = str(name).lower().rstrip(".")
        tokens.extend(token for token in TOKEN_SPLIT.split(normalized) if token)
    unique = list(dict.fromkeys(tokens))[:max_tokens]
    return " ".join(unique) if unique else MISSING


def _mode(values):
    clean = [str(value) for value in values
             if not _is_missing(value) and str(value).strip() not in ("", MISSING)]
    if not clean:
        return MISSING
    return pd.Series(clean).mode().iat[0]


def _tls_fingerprint(record):
    version = normalize_tls_version(record.get("version", record.get("tls_version")))
    ciphers = normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
    alpn = _text(record.get("alpn"), lowercase=True) or ""
    if MISSING in (version, ciphers):
        return MISSING
    return f"{version}|{ciphers}|{alpn}"

In [10]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def aggregate(records, feature_cols):
    """Aggregate canonical records for one device/window into one model feature row."""
    by_source = {source: [] for source in RAW_FIELDS}
    for record in records:
        if not isinstance(record, dict):
            raise TypeError("Each raw record must be a dictionary")
        proto = record.get("proto")
        if proto not in by_source:
            raise ValueError(f"Invalid proto {proto!r}; expected one of {sorted(RAW_FIELDS)}")
        by_source[proto].append(record)

    row = {}
    row["dhcp_prl"] = _mode(record.get("opt55", record.get("param_req_list"))
                             for record in by_source["dhcp"])
    row["dhcp_vci"] = _mode(record.get("opt60", record.get("vendor_class_id"))
                             for record in by_source["dhcp"])
    options = set()
    if row["dhcp_prl"] != MISSING:
        options = {int(value) for value in re.findall(r"\d+", row["dhcp_prl"])}
    for column in feature_cols:
        if column.startswith("dhcp_opt_"):
            row[column] = int(int(column.rsplit("_", 1)[1]) in options)
    row["dhcp_prl_len"] = len(options)

    row["dns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                         for record in by_source["dns"])
    row["mdns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                          for record in by_source["mdns"])

    tls_records = by_source["tls"]
    row["tls_fp"] = _mode(_tls_fingerprint(record) for record in tls_records)
    row["tls_version"] = _mode(
        normalize_tls_version(record.get("version", record.get("tls_version")))
        for record in tls_records
    )
    row["tls_alpn"] = _mode(_text(record.get("alpn"), lowercase=True)
                              for record in tls_records)
    row["tls_ciphers"] = _mode(
        normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
        for record in tls_records
    )
    row["tls_sni_tokens"] = domain_tokens(record.get("sni") for record in tls_records)

    for source in RAW_FIELDS:
        row[f"has_{source}"] = int(bool(by_source[source]))
    row["n_sources"] = sum(row[flag] for flag in SOURCE_FLAGS)

    numeric_prefixes = ("has_", "dhcp_opt_")
    for column in feature_cols:
        if column not in row:
            row[column] = (0 if column.startswith(numeric_prefixes)
                           or column == "dhcp_prl_len" else MISSING)
    return row

In [11]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Hằng của contract `sdc_iden.onnx` (09). 12/13 ghép head OS vào chính graph này nên dùng chung.
import hashlib
UNKNOWN_LABEL = "__unknown__"
NOT_PRESENT = ("", MISSING, "nan", "None")

In [12]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Chuẩn bị input 12 cột — bản Python của phần graph tự tính (`build_derived` ở 09).
import re

import numpy as np
import pandas as pd


_MAC_RE = re.compile(r"^[0-9a-f]{2}(:[0-9a-f]{2}){5}$")


def canonical_mac(value):
    if value is None:
        return MISSING
    text = str(value).strip().lower().replace("-", ":")
    return text if _MAC_RE.match(text) else MISSING


def mac_is_random(value):
    """Bit U/L (0x02) của octet đầu: 1 = locally administered = MAC ngẫu nhiên hoá."""
    mac = canonical_mac(value)
    return 0 if mac == MISSING else int(int(mac[:2], 16) & 0x02 != 0)

In [13]:
# --- Head OS (05 phần B) ---
import re


OS_CLASSES = ["iOS", "Android", "Windows", "macOS", "Linux", "Embedded"]
OS_THRESHOLD = 0.6          # ngưỡng của rừng (L2); L0/L1 là luật, conf = 1.0


def prl_set(prl):
    text = str(prl)
    return frozenset() if text in NOT_PRESENT else frozenset(int(t) for t in re.findall(r"\d+", text))


HOST_OS = [
    (r"^(iphone|ipad|ipod)\b", "iOS"),
    (r"^(macbook|imac|mac-?mini|mac-?pro|mac-?studio)\b", "macOS"),
    (r"^(android|galaxy|sm-|redmi|xiaomi|poco|mi-|realme|oppo|vivo|oneplus|pixel|huawei|honor|nokia|moto)", "Android"),
    (r"^(desktop|laptop|win)-[a-z0-9]{4,}$", "Windows"),
    (r"^(raspberrypi|ubuntu|debian|fedora|archlinux)\b", "Linux"),
    (r"^(esp[_-]|tuya|shelly|tasmota|wled)", "Embedded"),
]


def l0_os(hostname, vci, mac_random):
    """Chuỗi thiết bị tự khai."""
    h = str(hostname or "").strip().lower()
    if h and h not in ("<missing>", "nan", "none"):
        for pattern, os_ in HOST_OS:
            if re.search(pattern, h):
                return os_, "hostname"
    v = str(vci or "").strip().lower()
    if v.startswith("msft "):
        return "Windows", "vci"
    # android-dhcp-* cũng có ở loa Echo Studio (MAC thật) -> chỉ tin khi MAC ngẫu nhiên.
    if v.startswith("android-dhcp-") and mac_random:
        return "Android", "vci"
    return None, None


def l1_os(signatures, prl, mdns, mac_random, min_jaccard=0.9):
    """Vân tay giao thức: tập option DHCP gần nhất, rồi Apple Continuity qua mDNS."""
    s = prl_set(prl)
    if s:
        best, owners = 0.0, set()
        for key, value in signatures.items():
            j = len(s & key) / len(s | key)
            if j > best + 1e-9:
                best, owners = j, set(value)
            elif abs(j - best) <= 1e-9:
                owners |= value
        if best >= min_jaccard and len(owners) == 1:
            os_ = next(iter(owners))
            # Vân tay thiết bị gia dụng mà MAC ngẫu nhiên hoá: bằng chứng trái nhau, nhường L2.
            if not (os_ == "Embedded" and mac_random):
                return os_, f"prl~{best:.2f}"
    # Handoff/AirDrop: Apple TV/HomePod cũng quảng bá nhưng dùng MAC thật.
    m = str(mdns)
    if mac_random and ("_companion link" in m or "_rdlink" in m):
        return "iOS", "mdns"
    return None, None


def os_decide(bundle, frame, thr=OS_THRESHOLD, use_hostname=True):
    """-> list (nhãn, nguồn, conf) cho từng dòng."""
    X, _, _ = apply_encoder(frame[bundle["cols"]], bundle["encoder"])
    proba = bundle["model"].predict_proba(X)
    classes = bundle["model"].classes_
    out = []
    for i, (_, r) in enumerate(frame.iterrows()):
        rnd = int(r.get("mac_is_random", 0))
        label, src = l0_os(r.get("dhcp_hostname") if use_hostname else None, r.get("dhcp_vci"), rnd)
        if not label:
            label, src = l1_os(bundle["signatures"], r.get("dhcp_prl"), r.get("mdns_tokens"), rnd)
        if label:
            out.append((label, src, 1.0))
            continue
        k = int(proba[i].argmax()); p, conf = classes[k], float(proba[i, k])
        # Vân tay TLS là của TRÌNH DUYỆT (Chrome trên Windows = Chrome trên Linux), thứ tự
        # cipher của Apple xáo theo phiên: cửa sổ chỉ-TLS không đủ để nói một OS đa dụng.
        tls_only = not (r.get("has_dhcp") == 1 or r.get("has_mdns") == 1)
        if conf >= thr and not (p != "Embedded" and tls_only):
            out.append((p, "forest", conf))
        else:
            out.append((UNKNOWN_LABEL, "abstain", conf))
    return out

In [14]:
from pathlib import Path
import json
from datetime import datetime
from collections import Counter, OrderedDict

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
import re
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)

## A. Head iden — make / type / model

### A1. Tập train: 12 thiết bị FIELD làm lớp thật, phần còn lại làm nền `__unknown__`

Danh sách thiết bị FIELD lấy động từ cột `scenario` — không hardcode tên, để notebook
này vẫn đúng khi bạn capture thêm thiết bị/ngày FIELD mới.

In [15]:
sessions, feature_groups = load_sessions(SESSIONS_PATH)
features = feature_groups["all"]

is_field = sessions["scenario"].eq("FIELD")
FIELD_DEVICES = sorted(sessions.loc[is_field, "canonical_device"].unique())

print(f"Tổng {len(sessions)} dòng — {is_field.sum()} dòng / {len(FIELD_DEVICES)} thiết bị "
      f"FIELD làm lớp thật, {(~is_field).sum()} dòng CIC-2022 làm nền {UNKNOWN_LABEL!r}")
for head in LABEL_COLS:
    vc = sessions.loc[is_field, head].value_counts()
    print(f"  {head}: {len(vc)} lớp — {dict(vc)}")

Tổng 8189 dòng — 50 dòng / 12 thiết bị FIELD làm lớp thật, 8139 dòng CIC-2022 làm nền '__unknown__'
  make: 8 lớp — {'Generic Laptop': np.int64(22), 'Camera': np.int64(6), 'Linova/Linux': np.int64(6), 'Samsung': np.int64(6), 'Raspberry Pi': np.int64(3), 'Xiaomi': np.int64(3), 'Apple': np.int64(3), 'OPPO': np.int64(1)}
  type: 4 lớp — {'Laptop': np.int64(28), 'Smartphone': np.int64(13), 'IP Camera': np.int64(6), 'Single-board Computer': np.int64(3)}
  model: 11 lớp — {'Windows Desktop DELL': np.int64(11), 'Generic IP Camera': np.int64(6), 'Windows Laptop HP': np.int64(6), 'Linova Laptop HP': np.int64(6), 'Samsung Galaxy': np.int64(6), 'Windows Desktop HP': np.int64(5), 'Raspberry Pi': np.int64(3), 'iPhone': np.int64(3), 'Redmi Note 14 Pro': np.int64(2), 'OPPO A92': np.int64(1), 'Redmi Note 10': np.int64(1)}


### A2. Fit encoder + model (250 cây, giống `make_model` ở mục Thư viện)

Encoder vẫn fit trên **toàn bộ** dữ liệu (kể cả CIC-2022) để vocab TF-IDF phủ được cả
nền — fit riêng trên 50 dòng FIELD thì DNS/TLS token của CIC-2022 thành out-of-vocab hết,
"trông khác lạ" mất nghĩa.

Nhưng fit trên toàn bộ **kèm trần `max_features`** thì hỏng theo chiều ngược lại, và đó
là lý do model cũ không nhận được thiết bị lúc chạy thật: TF-IDF cắt vocab theo tần suất
toàn corpus, mà 8139/8189 dòng là CIC-2022, nên token nhận dạng của 12 thiết bị FIELD
thua cuộc đua tần suất và bị đá ra. Đo trên bản `20260915_170620_field_closedset`:

| thiết bị | `dns_tokens` OOV | `tls_sni_tokens` OOV | token bị mất |
|---|---|---|---|
| Raspberry Pi | 74% | 75% | `mozilla` `firefox` `detectportal` `safebrowsing` `merino` `push` `services` |
| Linova Laptop | 71% | 48% | `archlinux` `vntek` `vnpt` `sharepoint` `chatgpt`(dns) |

Những cái bị mất đúng là loại tín hiệu ổn định nhất: `detectportal.firefox.com` và
`ping.archlinux.org` là connectivity-check tự động của Firefox / NetworkManager, phát ra
mỗi lần lên mạng chứ không phụ thuộc người dùng bấm gì. Còn lại trong vocab chỉ là
`com` / `org` / `googleapis` — không đủ để phân biệt gì, nên forest chỉ còn cách nhớ
thuộc lòng đúng 3–6 dòng train.

Xem cell dưới cho hai fix và phần chúng **không** chữa được.

In [16]:
# --- FIX 1: bỏ trần `max_features` của TF-IDF -------------------------------------
# `_make_vectorizer` ở 03 đọc `TFIDF_MAX_FEATURES` tại thời điểm fit, nên ghi đè ở đây là
# đủ — không đụng vào contract `sdc-tiered-v2` mà 03/05/06/07 đang dùng chung.
#
# Trần 200/100/150/100 là hợp lý khi 12 lớp đều có hàng trăm phiên như ở 03. Ở đây tập
# FIELD chỉ 50 dòng, nên trần đó cắt đúng vào phần cần giữ (xem markdown trên). Bỏ trần
# chỉ nở input tensor 518 -> 1088 feature, vẫn thừa sức chạy trên router.
TFIDF_MAX_FEATURES = {col: None for col in TFIDF_MAX_FEATURES}

encoder = fit_encoder(sessions, features)
X_all, feature_names, _ = apply_encoder(sessions, encoder)
print(f"vocab sau khi bỏ trần: {X_all.shape[1]} feature — "
      + ", ".join(f"{c}={len(encoder['ct'].named_transformers_['t_' + c].vocabulary_)}"
                  for c in encoder["text_cols"]))

# --- FIX 2: không ép dòng non-FIELD mang nhãn thật thành `__unknown__` -------------
# `np.where(is_field, ..., UNKNOWN_LABEL)` gán TẤT CẢ dòng CIC-2022 thành `__unknown__`,
# kể cả những dòng mang đúng cái nhãn mà một thiết bị FIELD đang giữ. Head `type` dính
# nặng: 2471 dòng CIC-2022 có `type == "IP Camera"` thật, chọi 6 dòng FIELD cùng nhãn —
# forest được dạy hai điều ngược nhau trên cùng một nhãn, tỉ lệ 412:1.
#
# Bỏ hẳn các dòng đó khỏi tập train của head đang xét, thay vì dán nhãn sai cho chúng.
# Nền `__unknown__` của head `type` vẫn còn 5668 dòng nên khả năng từ chối không mất:
# đo lại trên CIC-2022 vẫn 0/8139 báo nhầm, IoT Sentinel vẫn 510/510 từ chối đúng.
models = {}
train_rows = {}
for head in LABEL_COLS:
    y = np.where(is_field, sessions[head].astype(str), UNKNOWN_LABEL)
    field_labels = set(sessions.loc[is_field, head].astype(str))
    clash = (~is_field) & sessions[head].astype(str).isin(field_labels)
    keep = ~clash.to_numpy()
    train_rows[head] = int(keep.sum())

    m = make_model()
    m.fit(X_all[keep], y[keep])
    models[head] = m
    print(f"  fit {head}: {len(m.classes_)} lop (gom {UNKNOWN_LABEL!r}), "
          f"{keep.sum()} dong train, bo {(~keep).sum()} dong non-FIELD trung nhan that")

# --- Hai fix này KHÔNG chữa được cái gì --------------------------------------------
# Không chữa được việc lớp chỉ có 1-6 phiên, tất cả từ MỘT ngày capture và MỘT MAC.
# `RARE_THRESHOLD = 10` ở 03 nói thẳng "lớp dưới ngưỡng này không học được"; notebook này
# đang train Raspberry Pi (3 phiên), iPhone (3), Redmi Note 14 Pro (2), OPPO A92 (1),
# Redmi Note 10 (1) bất chấp ngưỡng đó. Với `class_weight="balanced"` mỗi phiên Pi nặng
# gấp ~2700 lần một phiên `__unknown__`, nên forest khoét đúng 3 ô hẹp quanh 3 điểm đó.
#
# Riêng Linova Laptop còn một vấn đề dữ liệu mà không fix code nào chạm tới được: nó
# KHÔNG có feature nội tại nào khác 4 laptop FIELD còn lại — cùng `tls_fp`
# (0303|4865,4866,4867,...|h2,http/1.1, dùng chung với Desktop PC DNGJHRT / DucAnh /
# VAF70SQ6 / Lee Kingdom), không mDNS, không `dhcp_vci`, `dhcp_prl` trùng hệt Raspberry
# Pi. Thứ duy nhất tách nó ra là tập token DNS/SNI của người dùng. Đã thử bỏ trần +
# binary/norm=None: Linova vẫn `__unknown__` ngay khi đổi tập duyệt web.
print(f"\nSố phiên mỗi lớp (nhắc lại — RARE_THRESHOLD={RARE_THRESHOLD}):")
for head in LABEL_COLS:
    thin = sessions.loc[is_field, head].value_counts()
    thin = thin[thin < RARE_THRESHOLD]
    print(f"  {head}: {dict(thin)}")

vocab sau khi bỏ trần: 1088 feature — dhcp_vci=32, dns_tokens=575, mdns_tokens=118, tls_sni_tokens=327
  fit make: 9 lop (gom '__unknown__'), 8189 dong train, bo 0 dong non-FIELD trung nhan that
  fit type: 5 lop (gom '__unknown__'), 5718 dong train, bo 2471 dong non-FIELD trung nhan that
  fit model: 12 lop (gom '__unknown__'), 8189 dong train, bo 0 dong non-FIELD trung nhan that

Số phiên mỗi lớp (nhắc lại — RARE_THRESHOLD=10):
  make: {'Camera': np.int64(6), 'Linova/Linux': np.int64(6), 'Samsung': np.int64(6), 'Raspberry Pi': np.int64(3), 'Xiaomi': np.int64(3), 'Apple': np.int64(3), 'OPPO': np.int64(1)}
  type: {'IP Camera': np.int64(6), 'Single-board Computer': np.int64(3)}
  model: {'Generic IP Camera': np.int64(6), 'Windows Laptop HP': np.int64(6), 'Linova Laptop HP': np.int64(6), 'Samsung Galaxy': np.int64(6), 'Windows Desktop HP': np.int64(5), 'Raspberry Pi': np.int64(3), 'iPhone': np.int64(3), 'Redmi Note 14 Pro': np.int64(2), 'OPPO A92': np.int64(1), 'Redmi Note 10': np.int64

### A3. Lưu `iden.joblib`

Run mới `Models/<ts>/`, thư mục `train/`: `iden.joblib` (định dạng `sdc-closedset-v1`) và `meta.json`. Phần B ghi thêm
`os.joblib` vào **cùng thư mục**. Bundle iden không có bảng L1 hay ngưỡng theo `n_sources` — 09 đào và nung
những thứ đó lúc đóng gói; ở đây lớp `__unknown__` đã là cơ chế từ chối.

In [17]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = MODELS / RUN_ID / "train"   # 09 ghi iden/, 12–13 ghi release/ vào cùng Models/<RUN_ID>/
run_dir.mkdir(parents=True)

bundle = {
    "format": "sdc-closedset-v1",
    "unknown_label": UNKNOWN_LABEL,
    "heads": list(LABEL_COLS),
    "feature_cols": features,
    "feature_names": feature_names,
    "encoder": encoder,
    "models": models,
    "field_devices": FIELD_DEVICES,
}
joblib.dump(bundle, run_dir / "iden.joblib", compress=3)


def write_meta():
    (run_dir / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")


meta = {"format": "sdc-train-v1", "run_id": RUN_ID, "created": datetime.now().isoformat(timespec="seconds"),
        "dataset": str(SESSIONS_PATH), "n_sessions": int(len(sessions))}
meta["iden"] = {
    "format": "sdc-closedset-v1",
    "n_field_devices": len(FIELD_DEVICES),
    "field_devices": FIELD_DEVICES,
    # Đọc thẳng từ model đã fit, không dựng lại y để đếm: sau FIX 2 mỗi head có tập train
    # riêng, đếm lại bằng công thức cũ là ghi vào meta một con số không phải của model này.
    "heads": {h: int(len(models[h].classes_)) for h in LABEL_COLS},
    "train_rows": train_rows,
    "n_features": int(X_all.shape[1]),
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "note": ("Closed-set: 12 thiết bị FIELD là lớp thật, CIC-2022 làm nền __unknown__. "
             "FIX 1: bỏ trần TF-IDF max_features (trần cũ cắt mất 71-75% token nhận dạng "
             "của FIELD vì vocab bị 8139 dòng CIC-2022 chiếm chỗ). FIX 2: dòng non-FIELD "
             "mang đúng nhãn của một lớp FIELD bị loại khỏi tập train của head đó thay vì "
             "ép thành __unknown__ (head type: 2471 dòng IP Camera). "
             "CẢNH BÁO CHƯA GỠ: đánh giá FIELD ở notebook này là in-sample, 1 ngày capture, "
             "1 MAC mỗi thiết bị. Các lớp dưới RARE_THRESHOLD=10 (Raspberry Pi 3 phiên, "
             "iPhone 3, Redmi Note 14 Pro 2, OPPO A92 1, Redmi Note 10 1) là ghi nhớ thuộc "
             "lòng, không phải khái quát. Linova Laptop không có feature nội tại nào khác "
             "4 laptop FIELD còn lại (chung tls_fp, không mDNS, không dhcp_vci) — chỉ tách "
             "được bằng token DNS/SNI của người dùng, nên sẽ trượt lúc chạy thật."),
}
write_meta()
print("Đã lưu:", run_dir / "iden.joblib")

Đã lưu: D:\01.AI_Security\02.SDC\Models\20261013_094051\train\iden.joblib


### A4. Predictor tối giản cho head iden

Không cần bảng L1 hay ngưỡng theo nguồn bằng chứng — chỉ argmax `predict_proba`.
`is_unknown=True` khi lớp thắng là `__unknown__`.

In [18]:
class ClosedSetPredictor:
    def __init__(self, run_dir):
        bundle = joblib.load(Path(run_dir) / "iden.joblib")
        assert bundle["format"] == "sdc-closedset-v1"
        self.unknown_label = bundle["unknown_label"]
        self.heads = bundle["heads"]
        self.feature_cols = bundle["feature_cols"]
        self.encoder = bundle["encoder"]
        self.models = bundle["models"]
        self.field_devices = bundle["field_devices"]

    def predict_row(self, row):
        frame = pd.DataFrame([row])[self.feature_cols]
        X, _, _ = apply_encoder(frame, self.encoder)
        out = {}
        for head in self.heads:
            proba = self.models[head].predict_proba(X)[0]
            classes = self.models[head].classes_
            idx = proba.argmax()
            top1, conf = classes[idx], float(proba[idx])
            out[head] = {"top1": top1, "confidence": conf,
                         "is_unknown": top1 == self.unknown_label}
        return out

    def predict_device(self, rows):
        """Gộp nhiều phiên/cửa sổ của cùng một thiết bị bằng bỏ phiếu đa số trên top1
        (không cần ratio/ambiguous vì __unknown__ đã
        gánh vai trò từ chối)."""
        votes = {h: Counter() for h in self.heads}
        for row in rows:
            out = self.predict_row(row)
            for head in self.heads:
                votes[head][out[head]["top1"]] += 1
        result = {}
        for head in self.heads:
            top1, n = votes[head].most_common(1)[0]
            result[head] = {"top1": top1, "n_votes": n, "n_total": len(rows),
                             "is_unknown": top1 == self.unknown_label}
        return result


predictor = ClosedSetPredictor(run_dir)
print("Nạp lại từ", run_dir, "— OK")

Nạp lại từ D:\01.AI_Security\02.SDC\Models\20261013_094051\train — OK


### A5. Đánh giá

1. **Tự soi FIELD** — in-sample, chỉ để kiểm tra model học được (không phải bằng chứng
   khái quát, xem cảnh báo ở đầu notebook).
2. **IoT Sentinel** — 29 thiết bị hoàn toàn chưa đưa vào lúc fit, đây mới là phép thử
   trung thực cho khả năng từ chối "ngoài list".
3. **Biên độ xác suất** — kiểm tra thắng có sát nút không.

In [19]:
def eval_rows(sub_frame, label, is_field_data):
    Xs, _, _ = apply_encoder(sub_frame, predictor.encoder)
    print(f"\n=== {label} ({len(sub_frame)} dòng) ===")
    for head in predictor.heads:
        proba = predictor.models[head].predict_proba(Xs)
        classes = predictor.models[head].classes_
        top_idx = proba.argmax(axis=1)
        top1 = classes[top_idx]
        if is_field_data:
            truth = sub_frame[head].astype(str).to_numpy()
            correct = (top1 == truth).sum()
            to_unknown = (top1 == predictor.unknown_label).sum() - (truth == predictor.unknown_label).sum()
            other_wrong = len(sub_frame) - correct - to_unknown
            print(f"  {head:6s} đúng: {correct:4d}/{len(sub_frame)}   "
                  f"sai->__unknown__: {to_unknown:4d}   sai->lớp khác: {other_wrong:4d}")
        else:
            rejected = (top1 == predictor.unknown_label).sum()
            leaked = len(sub_frame) - rejected
            print(f"  {head:6s} từ chối đúng (__unknown__): {rejected:4d}/{len(sub_frame)}   "
                  f"BÁO NHẦM (gán vào 1 trong {len(FIELD_DEVICES)} thiết bị): {leaked:4d}")


eval_rows(sessions[is_field], "FIELD tự soi (in-sample — xem cảnh báo ở đầu notebook)",
          is_field_data=True)


=== FIELD tự soi (in-sample — xem cảnh báo ở đầu notebook) (50 dòng) ===
  make   đúng:   50/50   sai->__unknown__:    0   sai->lớp khác:    0
  type   đúng:   50/50   sai->__unknown__:    0   sai->lớp khác:    0
  model  đúng:   50/50   sai->__unknown__:    0   sai->lớp khác:    0


In [20]:
def load_windows(path):
    by_device = OrderedDict()
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            w = json.loads(line)
            by_device.setdefault(w["device"], []).append(w)
    return by_device


# IoT Sentinel chỉ có trên server; thiếu thì bỏ qua phép đo này, KHÔNG dừng notebook (phần B còn phải train).
SENTINEL_WINDOWS = ROOT / "test_model" / "data_test" / "out" / "windows.jsonl"
sentinel_frame = None
if SENTINEL_WINDOWS.is_file():
    by_device = load_windows(SENTINEL_WINDOWS)
    sentinel_frame = pd.DataFrame([aggregate(w["records"], features)
                                   for windows in by_device.values() for w in windows])
    eval_rows(sentinel_frame,
              f"IoT Sentinel ({len(by_device)} thiết bị lạ, chưa từng đưa vào lúc fit)",
              is_field_data=False)
else:
    print(f"Bỏ qua IoT Sentinel: không có {SENTINEL_WINDOWS} (dữ liệu chỉ có trên server)")

Bỏ qua IoT Sentinel: không có D:\01.AI_Security\02.SDC\test_model\data_test\out\windows.jsonl (dữ liệu chỉ có trên server)


In [21]:
def margin_report(sub_frame, label):
    Xs, _, _ = apply_encoder(sub_frame, predictor.encoder)
    print(f"\n=== Biên độ xác suất — {label} ===")
    for head in predictor.heads:
        proba = predictor.models[head].predict_proba(Xs)
        classes = predictor.models[head].classes_
        unk_idx = list(classes).index(predictor.unknown_label)
        p_unknown = proba[:, unk_idx]
        other_idx = [i for i in range(len(classes)) if i != unk_idx]
        p_best_other = proba[:, other_idx].max(axis=1)
        print(f"  {head:6s} p(__unknown__): min={p_unknown.min():.3f} mean={p_unknown.mean():.3f}   "
              f"p(lớp thật cao nhất): max={p_best_other.max():.3f} mean={p_best_other.mean():.3f}")


if sentinel_frame is not None:
    margin_report(sentinel_frame, "IoT Sentinel (ngoài list)")
margin_report(sessions[is_field], "FIELD (trong list, in-sample)")


=== Biên độ xác suất — FIELD (trong list, in-sample) ===
  make   p(__unknown__): min=0.000 mean=0.000   p(lớp thật cao nhất): max=1.000 mean=1.000
  type   p(__unknown__): min=0.000 mean=0.000   p(lớp thật cao nhất): max=1.000 mean=1.000
  model  p(__unknown__): min=0.000 mean=0.000   p(lớp thật cao nhất): max=1.000 mean=1.000


In [22]:
def verdict_of(pred, truth):
    if pred == truth:
        return "OK"
    if pred == predictor.unknown_label:
        return "BỎ SÓT"
    return "SAI"


rows = []
for device, group in sessions[is_field].groupby("canonical_device", sort=True):
    device_rows = [r.to_dict() for _, r in group.iterrows()]
    result = predictor.predict_device(device_rows)
    entry = {"device": device, "n_phien": len(device_rows)}
    for head in predictor.heads:
        truth = group[head].iloc[0]
        pred = result[head]["top1"]
        entry[f"{head}_pred"] = "" if pred == predictor.unknown_label else pred
        entry[f"{head}_truth"] = truth
        entry[f"{head}_verdict"] = verdict_of(pred, truth)
    rows.append(entry)

closedset_report = pd.DataFrame(rows)
closedset_summary = pd.DataFrame({
    head: closedset_report[f"{head}_verdict"].value_counts() for head in predictor.heads
}).T.fillna(0).astype(int)
display(closedset_summary)
display(closedset_report)

,OK
make,12
type,12
model,12


,device,n_phien,make_pred,make_truth,make_verdict,type_pred,type_truth,type_verdict,model_pred,model_truth,model_verdict
0,Desktop PC DNGJHRT,5,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop HP,Windows Desktop HP,OK
1,Desktop PC DucAnh,6,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop DELL,Windows Desktop DELL,OK
2,IP Camera (field),6,Camera,Camera,OK,IP Camera,IP Camera,OK,Generic IP Camera,Generic IP Camera,OK
3,Laptop VAF70SQ6,6,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Laptop HP,Windows Laptop HP,OK
4,Lee Kingdom Laptop,5,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop DELL,Windows Desktop DELL,OK
5,Linova Laptop (linova),6,Linova/Linux,Linova/Linux,OK,Laptop,Laptop,OK,Linova Laptop HP,Linova Laptop HP,OK
6,OPPO A92,1,OPPO,OPPO,OK,Smartphone,Smartphone,OK,OPPO A92,OPPO A92,OK
7,Raspberry Pi,3,Raspberry Pi,Raspberry Pi,OK,Single-board Computer,Single-board Computer,OK,Raspberry Pi,Raspberry Pi,OK
8,Samsung Phone (Duc Anh),6,Samsung,Samsung,OK,Smartphone,Smartphone,OK,Samsung Galaxy,Samsung Galaxy,OK
9,Xiaomi Redmi Note 10,1,Xiaomi,Xiaomi,OK,Smartphone,Smartphone,OK,Redmi Note 10,Redmi Note 10,OK


## B. Head OS — iOS / Android / Windows / macOS / Linux / Embedded

Thêm một tầng **trên** ba head cũ: `os` ∈ {iOS, Android, Windows, macOS, Linux, Embedded}.
Thứ tự trả lời **OS → type → make → model**, mỗi tầng tự abstain.

Lý do: ba head cũ chỉ biết 12 thiết bị FIELD, gặp thiết bị lạ là `__unknown__` hết. Nhưng
**OS lộ ra qua vân tay cấu trúc** (PRL, VCI, dịch vụ mDNS) và vân tay đó dùng chung cho
mọi máy cùng OS — `realme-6i` chưa từng train vẫn gửi đúng kiểu DHCP của Android.

| Tầng | Bằng chứng | Ghi chú |
|---|---|---|
| L0 | hostname mặc định (`iPhone`, `DESKTOP-…`, `realme-…`), VCI `MSFT 5.0`, VCI `android-dhcp-*` **+ MAC ngẫu nhiên** | chuỗi thiết bị tự khai |
| L1 | tập option DHCP gần nhất (Jaccard ≥ 0.9), dịch vụ Apple Continuity qua mDNS **+ MAC ngẫu nhiên** | tri thức giao thức, chịu được extractor sort lại PRL |
| L2 | rừng trên feature cấu trúc — **không** `dns_tokens`/`tls_sni_tokens` | chỉ được nói OS đa dụng khi cửa sổ có DHCP hoặc mDNS |

`Embedded` = firmware thiết bị chuyên dụng (camera, loa, ổ cắm, TV…) **kể cả khi nhân là
Linux/Android**: câu hỏi sản phẩm là "có phải máy tính đa dụng không". Echo Studio gửi
`android-dhcp-7.1.2` nhưng là loa — tách nó khỏi điện thoại bằng MAC: 5/5 điện thoại FIELD
ngẫu nhiên hoá MAC, 0/39 thiết bị CIC thì không.

**Giới hạn đã biết:** chưa có mẫu macOS nào (lớp chỉ trả lời được qua hostname); iOS chỉ 1
thiết bị train (L1-mDNS gánh phần tổng quát hoá); Linux 2 thiết bị.

### B1. Nhãn OS

FIELD gán theo bảng dưới; mọi thiết bị CIC-2022 là `Embedded`. Thiết bị FIELD mới mà chưa
có trong bảng thì **dừng hẳn** — đừng để nó lặng lẽ thành Embedded.

In [23]:
FIELD_OS = {
    "iPhone": "iOS",
    "OPPO A92": "Android", "Samsung Phone (Duc Anh)": "Android",
    "Xiaomi Redmi Note 10": "Android", "Xiaomi Redmi Note 14 Pro": "Android",
    "Desktop PC DNGJHRT": "Windows", "Desktop PC DucAnh": "Windows",
    "Laptop VAF70SQ6": "Windows", "Lee Kingdom Laptop": "Windows",
    "Linova Laptop (linova)": "Linux", "Raspberry Pi": "Linux",
    "IP Camera (field)": "Embedded",
}


sessions = sessions.copy()
is_field = sessions["scenario"].eq("FIELD")
unmapped = sorted(set(sessions.loc[is_field, "canonical_device"]) - set(FIELD_OS))
assert not unmapped, f"thiết bị FIELD chưa có nhãn OS: {unmapped}"
sessions["os"] = np.where(is_field, sessions["canonical_device"].map(FIELD_OS), "Embedded")
sessions["mac_is_random"] = sessions["mac"].map(mac_is_random)

summary = sessions.groupby("os").agg(thiết_bị=("canonical_device", "nunique"), cửa_sổ=("mac", "size"),
                                     mac_ngẫu_nhiên=("mac_is_random", "mean"))
display(summary.round(2))

,thiết_bị,cửa_sổ,mac_ngẫu_nhiên
os,,,
Android,4,10,1.0
Embedded,40,8145,0.0
Linux,2,9,0.0
Windows,4,22,0.0
iOS,1,3,1.0


### B2. Feature cấu trúc và cách cân mẫu

Bỏ `dns_tokens` và `tls_sni_tokens`: đó là *nội dung* (thiết bị gọi về đâu), đổi theo người
dùng — chính là thứ khiến head cũ học thuộc lòng. Thêm `mac_is_random`.

Cân **theo thiết bị**: mỗi thiết bị nặng như nhau rồi cân lớp — 331 cửa sổ của một ổ cắm
không được đè lên 3 cửa sổ của iPhone.

In [24]:
STRUCT_CAT = ["dhcp_prl", "tls_fp", "tls_version", "tls_alpn", "tls_ciphers"]
STRUCT_TEXT = ["dhcp_vci", "mdns_tokens"]


def os_feature_cols(df):
    num = [c for c in df.columns if c.startswith(("dhcp_opt_", "has_"))] + ["dhcp_prl_len"]
    return num + ["mac_is_random"] + STRUCT_CAT + STRUCT_TEXT


def device_weights(df, label="os"):
    w = 1.0 / df.groupby("canonical_device")["canonical_device"].transform("size")
    per_class = w.groupby(df[label]).transform("sum")
    return (w / per_class).to_numpy() * len(df) / df[label].nunique()


def mine_signatures(df):
    """Tập option DHCP -> tập OS đã thấy. TẬP chứ không chuỗi: extractor của đối tác sort
    lại thứ tự PRL (`1:108:114:...`) — so tập thì vẫn khớp."""
    sig = {}
    rows = df.loc[~df["dhcp_prl"].astype(str).isin(NOT_PRESENT), ["dhcp_prl", "os"]].drop_duplicates()
    for prl, os_ in rows.itertuples(index=False):
        sig.setdefault(prl_set(prl), set()).add(os_)
    return sig


def fit_os(df, cols):
    enc = fit_encoder(df, cols)
    X, _, _ = apply_encoder(df, enc)
    forest = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)
    forest.fit(X, df["os"].to_numpy(), sample_weight=device_weights(df))
    return {"encoder": enc, "model": forest, "cols": cols, "signatures": mine_signatures(df)}
OS_COLS = os_feature_cols(sessions)
print(f"{len(OS_COLS)} cột cấu trúc")

39 cột cấu trúc


### B3. Quyết định OS: L0 → L1 → L2

`HOST_OS`, `l0_os`, `l1_os`, `os_decide` nằm ở mục **Thư viện** đầu notebook. 12 và 13 có bản sao của đúng các hàm này — sửa luật thì sửa cả ba nơi (bảng *Hàm có bản sao ở đâu* trong README).

### B4. Cascade OS → type → make → model

* type/make **mâu thuẫn** OS -> abstain tầng đó (vd. OS=Windows mà type=Smartphone).
* tầng dưới **abstain** mà OS đủ nói thay -> điền: iOS → Smartphone + Apple;
  Android **+ MAC ngẫu nhiên** → Smartphone; Windows/macOS → Laptop; macOS → Apple.
* make hoặc type abstain thì model cũng abstain.

`TYPE_OK`, `MAKE_OK`, `cascade` nằm ở mục **Thư viện** đầu notebook (12 và 13 có bản sao).

### B5. Đánh giá trung thực: giữ lại theo **thiết bị**

OS phải tổng quát hoá sang thiết bị chưa thấy, nên mỗi thiết bị FIELD là một fold, thiết bị
CIC chia 6 nhóm. Encoder, rừng **và bảng chữ ký L1** đều dựng lại trên phần train của fold.
Không dùng hostname — đo phần học được, không đo phần tự khai.

In [25]:
cic_devices = sorted(sessions.loc[~is_field, "canonical_device"].unique())
rng = np.random.default_rng(0)
group = {d: f"cic{k}" for d, k in zip(cic_devices, rng.permutation(np.arange(len(cic_devices)) % 6))}
group.update({d: d for d in sessions.loc[is_field, "canonical_device"].unique()})
fold = sessions["canonical_device"].map(group).to_numpy()

oof_label = np.empty(len(sessions), dtype=object)
oof_src = np.empty(len(sessions), dtype=object)
for g in np.unique(fold):
    train, test = fold != g, fold == g
    b = fit_os(sessions[train], OS_COLS)
    for i, (label, src, _) in zip(np.where(test)[0], os_decide(b, sessions[test], use_hostname=False)):
        oof_label[i], oof_src[i] = label, src.split("~")[0]
sessions["os_pred"], sessions["os_src"] = oof_label, oof_src

rows = []
for os_ in OS_CLASSES:
    m = sessions["os"].eq(os_)
    if not m.any():
        continue
    ok = (m & sessions.os_pred.eq(os_)).sum()
    bad = (m & sessions.os_pred.ne(os_) & sessions.os_pred.ne(UNKNOWN_LABEL)).sum()
    rows.append({"os": os_, "thiết bị": sessions.loc[m, "canonical_device"].nunique(),
                 "cửa sổ": int(m.sum()), "đúng": int(ok), "sai": int(bad), "abstain": int(m.sum() - ok - bad)})
display(pd.DataFrame(rows).set_index("os"))
print("nguồn quyết định:", sessions.os_src.value_counts().to_dict())


def vote(g):
    answered = g[g.os_pred.ne(UNKNOWN_LABEL)]
    return answered.os_pred.mode().iat[0] if len(answered) else UNKNOWN_LABEL


by_device = sessions.groupby("canonical_device").apply(
    lambda g: pd.Series({"os": g.os.iat[0], "đoán": vote(g), "cửa sổ": len(g),
                         "đã trả lời": int(g.os_pred.ne(UNKNOWN_LABEL).sum())}), include_groups=False)
dev_ok = int(by_device.os.eq(by_device["đoán"]).sum())
dev_bad = int((by_device["đoán"].ne(UNKNOWN_LABEL) & by_device.os.ne(by_device["đoán"])).sum())
print(f"\nMức thiết bị (bỏ phiếu các cửa sổ đã trả lời): đúng {dev_ok}/{len(by_device)}, sai {dev_bad}")
display(by_device[by_device.os.ne("Embedded") | by_device["đoán"].ne("Embedded")])

,thiết bị,cửa sổ,đúng,sai,abstain
os,,,,,
iOS,1,3,3,0,0
Android,4,10,4,0,6
Windows,4,22,5,0,17
Linux,2,9,4,0,5
Embedded,40,8145,7648,8,489


nguồn quyết định: {'forest': 6878, 'prl': 782, 'abstain': 517, 'vci': 9, 'mdns': 3}

Mức thiết bị (bỏ phiếu các cửa sổ đã trả lời): đúng 48/51, sai 0


,os,đoán,cửa sổ,đã trả lời
canonical_device,,,,
Desktop PC DNGJHRT,Windows,Windows,5,1
Desktop PC DucAnh,Windows,Windows,6,3
LG TV,Embedded,__unknown__,3,0
Laptop VAF70SQ6,Windows,__unknown__,6,0
Lee Kingdom Laptop,Windows,Windows,5,1
Linova Laptop (linova),Linux,Linux,6,2
OPPO A92,Android,Android,1,1
Raspberry Pi,Linux,Linux,3,2
Samsung Phone (Duc Anh),Android,Android,6,2


### B6. Train bản cuối, lưu `os.joblib`

In [26]:
os_bundle = fit_os(sessions, OS_COLS)
joblib.dump({"format": "sdc-os-v0", "classes": OS_CLASSES, "threshold": OS_THRESHOLD,
             "field_os": FIELD_OS, **os_bundle}, run_dir / "os.joblib", compress=3)
meta["os"] = {
    "format": "sdc-os-v0", "classes": OS_CLASSES, "threshold": OS_THRESHOLD, "feature_cols": OS_COLS,
    "n_signatures": len(os_bundle["signatures"]),
    "oof_device": {"đúng": dev_ok, "sai": dev_bad, "tổng": int(len(by_device))},
    "note": "Thử nghiệm. Chưa có mẫu macOS; iOS 1 thiết bị, Linux 2.",
}
write_meta()
print("Đã lưu:", run_dir / "os.joblib")

Đã lưu: D:\01.AI_Security\02.SDC\Models\20261013_094051\train\os.joblib


## C. Train chung Data_AKien + hostname tự thu — cascade A → B → unknown

Mỗi head `make/type/model` chạy độc lập: luật hostname A khớp thì lấy kết quả luật; không khớp thì hỏi expert B; B không đạt threshold thì trả `__unknown__`. Kết quả kèm nguồn quyết định và confidence để kiểm tra.

Expert mặc định học từ **hai nguồn hợp nhất**: CSV `Data_AKien/Device_finger print/host_name_tutorial/data/all_data_1604.csv` và hostname DHCP tự thu nối với `Data/sessions_verified.parquet` + `Data/device_labels_verified.csv`. Không lấy tên hiển thị hoặc dự đoán model làm nhãn thật. Bản ghi chưa có nhãn verified bị loại; hãng giả như `Generic Laptop`, `Camera`, `Linova/Linux` được bỏ ở head make nhưng vẫn giữ nhãn type/model hợp lệ. `UA_data.csv` thiếu nhãn make/type/model; các SQLite chứa taxonomy nên chưa dùng làm mẫu supervised.

Phần C có thể chạy riêng từ cell cấu hình bên dưới, hoặc chạy cùng A/B của notebook 05 để lưu vào cùng run. Báo cáo so sánh **A_rules / B_expert / C_cascade** trên cùng test, đồng thời đo riêng theo nguồn AKIEN/FIELD. Thiết bị FIELD và mọi template hostname liên quan ở cùng một split; hostname trùng hai nguồn chỉ là một mẫu. Nếu nhãn xung đột, chỉ mask head xung đột và ghi audit, không tự chọn nhãn thắng. Threshold của expert chỉ chọn trên validation ở các hostname mà luật của head đó không khớp. Luật A là heuristics hiện tại; độ chính xác của chúng cũng được đo, không giả định mọi hit đều đúng.

Expert make chỉ được trả lời khi hostname có dấu hiệu hãng/family tương ứng. Trong pipeline gói tin, B chỉ bổ sung cho head đã abstain; kết quả gói tin đã nhận diện được được giữ lại. Model B mâu thuẫn type đã biết sẽ bị loại trước khi áp hierarchy.


In [2]:
# C1 — cấu hình riêng để tránh ghi đè biến của head iden/OS
import ast
import hashlib
import json
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit

HN_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if HN_ROOT is None:
    raise RuntimeError("Không tìm thấy project root chứa Code/ và Data/")
HN_DATA = HN_ROOT / "Data_AKien" / "Device_finger print"
HN_CSV = HN_DATA / "host_name_tutorial" / "data" / "all_data_1604.csv"
HN_RULE_NOTEBOOK = HN_ROOT / "Code/pipeline/out_model/09_build_sdc_iden.ipynb"
# Danh sách thiết bị cho rule A hostname (cột: docs/HOSTNAME_CATALOG_RULES.md). Thêm/sửa dòng
# trong file này rồi chạy lại 05 (phần C) → 09 → 12 → 13; rule được sinh lại ở C2 mỗi lần chạy.
HN_DEVICE_CATALOG = HN_ROOT / "Code/pipeline/device_list.csv"
HN_DATA_MODE = "combined"   # combined = AKIEN + FIELD; akien_only = mốc so sánh
HN_FIELD_SESSIONS = HN_ROOT / "Data/sessions_verified.parquet"
HN_FIELD_LABELS = HN_ROOT / "Data/device_labels_verified.csv"
HN_FIELD_DHCP = [HN_ROOT / "Data/features/dhcp_features_capture.csv"]
# Thêm file DHCP đã extract khác vào HN_FIELD_DHCP; cần MAC/capture trong sessions verified.
# Không dùng device_observations.jsonl làm train: đang là bộ test, chưa có bảng nhãn đầy đủ.
HN_FIELD_WEIGHT = 1.0       # trọng số FIELD; không nhân bản packet/mẫu để vượt MIN_TRAIN
assert HN_DATA_MODE in {"combined", "akien_only"}
assert np.isfinite(HN_FIELD_WEIGHT) and HN_FIELD_WEIGHT > 0
HN_HEADS = ("make", "type", "model")
HN_UNKNOWN = "__unknown__"
HN_SEED = 42
HN_MIN_TRAIN = 25
HN_MAX_FEATURES = 60_000
HN_TARGET_PRECISION = 0.90
HN_MIN_VALID_ANSWERS = 30
HN_BATCH_SIZE = 4096       # giới hạn bộ nhớ predict_proba cho head model nhiều class
HN_SAVE = True            # lưu artifact thử nghiệm, trạng thái gate đi kèm
HN_NULLS = {"", "null", "none", "nan", "<missing>", "<na>", "__unknown__"}
assert HN_CSV.is_file(), HN_CSV
assert HN_DEVICE_CATALOG.is_file(), HN_DEVICE_CATALOG
import sys
sys.path.insert(0, str(HN_ROOT / "Code" / "pipeline"))
from hostname_support import make_evidence_patterns, candidate_make_supported
from hostname_catalog import build as build_catalog_rules
from hostname_data import (load_akien_hostnames, load_field_hostnames,
                           prepare_hostname_dataset, split_hostname_dataset)

def hn_normalize(value):
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).strip().lower()
    return "" if text in HN_NULLS else re.sub(r"\s+", " ", text)

def hn_template(value):
    text = hn_normalize(value)
    text = re.sub(r"(?:[0-9a-f]{2}[:-]){5}[0-9a-f]{2}", "<mac>", text)
    text = re.sub(r"\b[0-9a-f]{8}-[0-9a-f-]{27,}\b", "<uuid>", text)
    text = re.sub(r"\b[0-9a-f]{10,}\b", "<hex>", text)
    return re.sub(r"\d+", "<n>", text)

def hn_label(value):
    return pd.NA if hn_normalize(value) == "" else str(value).strip()

def hn_read_rules(path):
    # Chỉ đọc literal list bằng AST; không thực thi notebook export.
    document = json.loads(path.read_text(encoding="utf-8"))
    for cell in document["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell.get("source", []))
        if "DEFAULT_HOSTNAME_RULES =" not in source:
            continue
        for node in ast.parse(source).body:
            if isinstance(node, ast.Assign) and any(
                isinstance(t, ast.Name) and t.id == "DEFAULT_HOSTNAME_RULES" for t in node.targets
            ):
                rules = ast.literal_eval(node.value)
                for pattern, head, label in rules:
                    assert head in HN_HEADS
                    re.compile(pattern)
                return rules
    raise RuntimeError(f"Không tìm thấy DEFAULT_HOSTNAME_RULES: {path}")

print("Nguồn AKIEN:", HN_CSV)
print("Chế độ train:", HN_DATA_MODE)
if HN_DATA_MODE == "combined":
    print("Nguồn FIELD:", HN_FIELD_DHCP)
    print("Nhãn xác thực:", HN_FIELD_LABELS)
print("Catalog thiết bị:", HN_DEVICE_CATALOG)


Nguồn AKIEN: D:\01.AI_Security\02.SDC\Data_AKien\Device_finger print\host_name_tutorial\data\all_data_1604.csv
Chế độ train: combined
Nguồn FIELD: [WindowsPath('D:/01.AI_Security/02.SDC/Data/features/dhcp_features_capture.csv')]
Nhãn xác thực: D:\01.AI_Security\02.SDC\Data\device_labels_verified.csv
Catalog thiết bị: D:\01.AI_Security\02.SDC\Code\pipeline\device_list.csv


In [3]:
# C2 — hợp nhất nguồn, audit nhãn, dedup và split theo component template + thiết bị FIELD
hn_akien = load_akien_hostnames(HN_CSV)
hn_field_load_audit = {}
if HN_DATA_MODE == "combined":
    hn_field, hn_field_load_audit = load_field_hostnames(
        HN_FIELD_SESSIONS, HN_FIELD_LABELS, HN_FIELD_DHCP)
else:
    hn_field = hn_akien.iloc[:0].copy()
hn_data, hn_conflict_audit, hn_data_audit = prepare_hostname_dataset(hn_akien, hn_field)
if HN_DATA_MODE == "combined" and hn_data_audit["field_unique_hostnames"] == 0:
    raise RuntimeError("Không tìm được hostname FIELD có nhãn verified; kiểm tra nguồn DHCP/MAC/capture.")
hn_splits = split_hostname_dataset(hn_data, seed=HN_SEED)
hn_train, hn_valid, hn_test = (hn_splits[name] for name in ("train", "validation", "test"))
if HN_DATA_MODE == "combined" and not hn_train["source_members"].str.contains("FIELD", regex=False).any():
    raise RuntimeError("Split này không có FIELD trong train; đổi HN_SEED hoặc bổ sung thiết bị verified.")
hn_split_manifest = pd.concat([frame.assign(split=name) for name, frame in hn_splits.items()], ignore_index=True)
hn_field_manifest = hn_split_manifest[
    hn_split_manifest["source_members"].str.contains("FIELD", regex=False)].copy()
hn_split_summary = pd.DataFrame([
    {"split": name, "rows": len(frame), "groups": frame["group_id"].nunique(),
     "AKIEN_rows": int(frame["source"].eq("AKIEN").sum()),
     "FIELD_rows": int(frame["source"].eq("FIELD").sum()),
     **{h + "_classes": frame[h].nunique() for h in HN_HEADS}}
    for name, frame in hn_splits.items()
])
display(hn_split_summary)
display(hn_field_manifest[["split", "hostname", "canonical_device", "make", "type", "model", "source_members"]])
print("Audit load FIELD:", hn_field_load_audit)
print("Audit hợp nhất:", hn_data_audit)
if not hn_conflict_audit.empty:
    display(hn_conflict_audit)
print("Leak check: hostname/template/group/thiết bị FIELD không trùng giữa train-validation-test.")
print("Lưu ý: held-out ở đây dành cho hostname expert; rừng fingerprint nền có thể đã học thiết bị FIELD.")

# Rule A: 09 trước, catalog sau (first-match-wins). Rule catalog chấm trên train+validation của
# chính split này; rule sai nhiều bị loại (xem hn_catalog_report). Nhãn __unknown__ là rule chặn:
# khớp thì head đó dừng ở A, không gọi expert.
hn_catalog, hn_catalog_report = build_catalog_rules(HN_DEVICE_CATALOG, hn_split_manifest, HN_RULE_NOTEBOOK)
hn_rules = hn_read_rules(HN_RULE_NOTEBOOK) + hn_catalog["rules"]
display(hn_catalog_report.groupby(["head", "status"]).size().unstack(fill_value=0))
for hint in hn_catalog["label_hints"]:
    print("Nhãn viết khác dataset:", hint)
print("Luật A:", HN_RULE_NOTEBOOK.name, "+", HN_DEVICE_CATALOG.name, "—", len(hn_rules), "luật")

def hn_rule_predictions(hostnames, head, rules):
    labels = np.full(len(hostnames), HN_UNKNOWN, dtype=object)
    patterns = np.full(len(hostnames), "", dtype=object)
    compiled = [(re.compile(p), y, p) for p, h, y in rules if h == head]
    for i, hostname in enumerate(hostnames):
        if not hostname:
            continue
        for pattern, label, raw_pattern in compiled:
            if pattern.fullmatch(hostname):
                labels[i], patterns[i] = label, raw_pattern
                break
    return labels, patterns


,split,rows,groups,AKIEN_rows,FIELD_rows,make_classes,type_classes,model_classes
0,train,165453,108196,165447,6,26,8,270
1,validation,39612,27050,39610,2,28,9,260
2,test,55964,33812,55963,1,26,8,255


,split,hostname,canonical_device,make,type,model,source_members
0,train,linova,Linova Laptop (linova),NaN,Laptop,Linova Laptop HP,FIELD
1,train,desktop-ducanh,Desktop PC DucAnh,NaN,Laptop,Windows Desktop DELL,FIELD
2,train,iphone,iPhone,Apple,Smartphone,iPhone,AKIEN|FIELD
3,train,redmi-note-14-pro,Xiaomi Redmi Note 14 Pro,Xiaomi,Smartphone,Redmi Note 14 Pro,FIELD
4,train,desktop-dngjhrt,Desktop PC DNGJHRT,NaN,Laptop,Windows Desktop HP,FIELD
5,train,samsung-cua-duc-anh,Samsung Phone (Duc Anh),Samsung,Smartphone,Samsung Galaxy,FIELD
165453,validation,raspberrypi,Raspberry Pi,Raspberry Pi,Single-board Computer,Raspberry Pi,FIELD
165454,validation,oppo-a92,OPPO A92,OPPO,Smartphone,OPPO A92,AKIEN|FIELD
205065,test,lee_kingdom,Lee Kingdom Laptop,NaN,Laptop,Windows Desktop DELL,FIELD


Audit load FIELD: {'dhcp_raw_rows': 41, 'unmatched_or_unreviewed_rows': 4, 'matched_packet_rows': 37, 'verified_devices': 12}
Audit hợp nhất: {'input_rows_by_source': {'AKIEN': 265641, 'FIELD': 37}, 'invalid_hostname_rows': 0, 'clean_rows': 261029, 'clean_rows_by_source': {'AKIEN': 261020, 'FIELD': 9}, 'field_unique_hostnames': 9, 'cross_source_shared_hostnames': 2, 'conflicting_heads': 9, 'removed_all_labels_missing_hostnames': 0, 'duplicate_hostname_rows_collapsed': 4649}


,hostname,head,labels,sources,action
0,a32-cua-dang,make,"[""OPPO"", ""Samsung""]",AKIEN,mask_conflicting_head
1,a12-cua-tinh,make,"[""OPPO"", ""Samsung""]",AKIEN,mask_conflicting_head
2,a31-cua-lam,make,"[""OPPO"", ""Samsung""]",AKIEN,mask_conflicting_head
3,a12-cua-tung,make,"[""OPPO"", ""Samsung""]",AKIEN,mask_conflicting_head
4,a32-cua-dang,model,"[""OPPO A32"", ""Samsung Galaxy A32""]",AKIEN,mask_conflicting_head
5,oppo-a17kv1,model,"[""OPPO 1105"", ""OPPO A17k""]",AKIEN,mask_conflicting_head
6,a12-cua-tinh,model,"[""OPPO A12"", ""Samsung Galaxy A12""]",AKIEN,mask_conflicting_head
7,a31-cua-lam,model,"[""OPPO A31"", ""Samsung Galaxy A31""]",AKIEN,mask_conflicting_head
8,a12-cua-tung,model,"[""OPPO A12"", ""Samsung Galaxy A12""]",AKIEN,mask_conflicting_head


Leak check: hostname/template/group/thiết bị FIELD không trùng giữa train-validation-test.
Lưu ý: held-out ở đây dành cho hostname expert; rừng fingerprint nền có thể đã học thiết bị FIELD.


status,rejected,rejected_row,stop,unverified,verified
head,,,,,
make,33,1,1,888,141
model,51,3,1,825,181
type,2,31,0,812,170


Luật A: 09_build_sdc_iden.ipynb + device_list.csv — 981 luật


### C3. Expert và ngưỡng cho fallback

TF-IDF ký tự + ba classifier. Class ít hơn 25 **hostname train duy nhất** không được học. FIELD có nhãn hiếm vẫn nằm trong dataset/audit nhưng không tự được nhân bản để vượt ngưỡng này; bảng class support ghi rõ mẫu nào thực sự được học. Mặc định trọng số FIELD = 1 như AKIEN; chỉ train trên phần train của cả hai nguồn. Với mỗi head, hiệu chỉnh threshold chỉ trên phần validation **không khớp luật A**, chọn coverage cao nhất đạt precision ≥ 90% và ít nhất 30 câu trả lời. Nếu không tìm được ngưỡng đạt yêu cầu, threshold = 1.01 để tắt fallback head đó. Không hạ threshold theo kết quả test. Metrics gộp và metrics FIELD được báo cáo riêng vì tập FIELD hiện nhỏ, không coi điểm cao trên AKIEN là chứng minh chất lượng trên FIELD.

Confidence là score của classifier, chưa được hiệu chỉnh thành xác suất đúng thực tế. Hostname trống hoặc không có bất kỳ n-gram nào trong vocabulary luôn abstain.


In [4]:
hn_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.995,
    max_features=HN_MAX_FEATURES, sublinear_tf=True, dtype=np.float32,
)
hn_X_train = hn_vectorizer.fit_transform(hn_train["hostname"])
hn_X_valid = hn_vectorizer.transform(hn_valid["hostname"])

def hn_expert_top1(model, X):
    labels = np.full(X.shape[0], HN_UNKNOWN, dtype=object)
    confidence = np.zeros(X.shape[0], dtype=float)
    for start in range(0, X.shape[0], HN_BATCH_SIZE):
        stop = min(start + HN_BATCH_SIZE, X.shape[0])
        proba = model.predict_proba(X[start:stop])
        idx = proba.argmax(axis=1)
        labels[start:stop] = model.classes_[idx]
        confidence[start:stop] = proba[np.arange(stop - start), idx]
    return labels, confidence

hn_experts = {}
hn_threshold_rows = []
hn_curves = []
hn_class_support_rows = []
for hn_head in HN_HEADS:
    hn_counts = hn_train[hn_head].value_counts()
    hn_supported = set(hn_counts[hn_counts >= HN_MIN_TRAIN].index)
    hn_field_counts = hn_train.loc[hn_train["source"].eq("FIELD"), hn_head].value_counts()
    for hn_label_name, hn_count in hn_counts.items():
        hn_class_support_rows.append({
            "head": hn_head, "label": str(hn_label_name), "unique_train_hostnames": int(hn_count),
            "FIELD_train_hostnames": int(hn_field_counts.get(hn_label_name, 0)),
            "AKIEN_train_hostnames": int(hn_count - hn_field_counts.get(hn_label_name, 0)),
            "supported": hn_label_name in hn_supported,
        })
    hn_keep = hn_train[hn_head].notna() & hn_train[hn_head].isin(hn_supported)
    if len(hn_supported) < 2:
        raise RuntimeError(f"{hn_head}: cần ít nhất 2 class đủ mẫu train")
    hn_model = SGDClassifier(
        loss="log_loss", penalty="l2", alpha=1e-5, class_weight="balanced",
        max_iter=50, tol=1e-3, average=True, random_state=HN_SEED,
    )
    hn_weights = np.where(hn_train.loc[hn_keep, "source"].eq("FIELD"), HN_FIELD_WEIGHT, 1.0)
    hn_model.fit(hn_X_train[hn_keep.to_numpy()], hn_train.loc[hn_keep, hn_head].astype(str),
                 sample_weight=hn_weights)
    hn_top, hn_conf = hn_expert_top1(hn_model, hn_X_valid)
    _, hn_rule_pattern = hn_rule_predictions(hn_valid["hostname"].tolist(), hn_head, hn_rules)
    hn_truth = hn_valid[hn_head].fillna("__missing_truth__").astype(str).to_numpy()
    hn_eval_mask = hn_valid[hn_head].notna().to_numpy() & (hn_rule_pattern == "")
    hn_nonzero = np.asarray(hn_X_valid.getnnz(axis=1)).ravel() > 0
    if hn_head == "make":
        hn_nonzero &= candidate_make_supported(hn_valid["hostname"].tolist(), hn_top,
            make_evidence_patterns({"experts": {"make": {"model": hn_model}}}))
    hn_curve = []
    for threshold in np.round(np.linspace(0.30, 0.98, 35), 2):
        answered = hn_eval_mask & hn_nonzero & (hn_conf >= threshold)
        n = int(answered.sum())
        precision = float(((hn_top == hn_truth) & answered).sum() / n) if n else np.nan
        hn_curve.append({"head": hn_head, "threshold": float(threshold), "answered": n,
                         "precision": precision,
                         "coverage": n / max(int(hn_eval_mask.sum()), 1)})
    hn_curve = pd.DataFrame(hn_curve)
    hn_feasible = hn_curve[
        (hn_curve["precision"] >= HN_TARGET_PRECISION)
        & (hn_curve["answered"] >= HN_MIN_VALID_ANSWERS)
    ].sort_values(["coverage", "threshold"], ascending=[False, True])
    hn_best = hn_feasible.iloc[0].to_dict() if len(hn_feasible) else {
        "head": hn_head, "threshold": 1.01, "answered": 0,
        "precision": np.nan, "coverage": 0.0,
    }
    hn_best["fallback_enabled"] = bool(len(hn_feasible))
    hn_best["validation_rule_miss_rows"] = int(hn_eval_mask.sum())
    hn_threshold_rows.append(hn_best)
    hn_curves.append(hn_curve)
    hn_experts[hn_head] = {"model": hn_model, "threshold": float(hn_best["threshold"])}
    print(f"{hn_head}: {int(hn_keep.sum()):,} train rows, {len(hn_model.classes_)} classes, "
          f"fallback threshold={hn_best['threshold']:.2f}, "
          f"FIELD actually fitted={int((hn_keep & hn_train['source'].eq('FIELD')).sum())}")
hn_thresholds = pd.DataFrame(hn_threshold_rows).set_index("head")
hn_class_support = pd.DataFrame(hn_class_support_rows)
display(hn_thresholds)
display(hn_class_support[hn_class_support["FIELD_train_hostnames"].gt(0)])
print("FIELD có nhãn nhưng class chưa đủ mẫu sẽ không được fit cho head đó; không suy nhãn thay thế.")


make: 63,059 train rows, 20 classes, fallback threshold=0.30, FIELD actually fitted=3


type: 165,453 train rows, 8 classes, fallback threshold=0.30, FIELD actually fitted=6


model: 61,849 train rows, 179 classes, fallback threshold=0.30, FIELD actually fitted=1


,threshold,answered,precision,coverage,fallback_enabled,validation_rule_miss_rows
head,,,,,,
make,0.3,444,1.000000,0.079842,True,5561
type,0.3,7137,0.999159,1.000000,True,7137
model,0.3,5349,0.982987,0.974850,True,5487


,head,label,unique_train_hostnames,FIELD_train_hostnames,AKIEN_train_hostnames,supported
0,make,Samsung,31873,1,31872,True
2,make,Apple,6202,1,6201,True
3,make,Xiaomi,5931,1,5930,True
26,type,Laptop,58324,3,58321,True
27,type,Smartphone,43655,3,43652,True
35,model,iPhone,3269,1,3268,True
298,model,Linova Laptop HP,1,1,0,False
299,model,Windows Desktop DELL,1,1,0,False
300,model,Redmi Note 14 Pro,1,1,0,False
301,model,Windows Desktop HP,1,1,0,False


FIELD có nhãn nhưng class chưa đủ mẫu sẽ không được fit cho head đó; không suy nhãn thay thế.


In [5]:
# C4 — API cascade: expert chỉ được gọi với các dòng luật của head đó không khớp
class HostnameCascade:
    def __init__(self, bundle):
        if bundle.get("format") != "sdc-hostname-cascade-v1":
            raise ValueError("Sai format hostname cascade")
        self.bundle = bundle
        self.make_evidence = make_evidence_patterns(bundle)

    @classmethod
    def load(cls, path):
        return cls(joblib.load(path))

    def predict(self, hostnames):
        if isinstance(hostnames, str) or hostnames is None:
            hostnames = [hostnames]
        hosts = [hn_normalize(value) for value in hostnames]
        n = len(hosts)
        valid = np.array([bool(h) and len(h) <= 253 for h in hosts], dtype=bool)
        hosts_for_rules = [h if ok else "" for h, ok in zip(hosts, valid)]
        frames = []
        for head in HN_HEADS:
            item = self.bundle["experts"][head]
            threshold = float(item["threshold"])
            labels, patterns = hn_rule_predictions(hosts_for_rules, head, self.bundle["rules"])
            # Khớp pattern là trúng A, kể cả rule chặn __unknown__ (giống hostname_reference ở 12/13).
            rule_hit = patterns != ""
            sources = np.where(rule_hit, "A_rule", "abstain").astype(object)
            confidence = np.where(rule_hit, 1.0, 0.0)
            reasons = np.where(rule_hit, "rule_match", "invalid_hostname").astype(object)
            reasons[rule_hit & (labels == HN_UNKNOWN)] = "rule_stop"
            candidate = np.full(n, HN_UNKNOWN, dtype=object)
            called = np.zeros(n, dtype=bool)
            misses = np.flatnonzero(valid & ~rule_hit)
            if len(misses):
                reasons[misses] = "expert_disabled"
            if len(misses) and threshold <= 1.0:
                X = self.bundle["vectorizer"].transform([hosts[i] for i in misses])
                usable = np.asarray(X.getnnz(axis=1)).ravel() > 0
                reasons[misses] = "no_known_ngram"
                positions = misses[usable]
                if len(positions):
                    top, conf = hn_expert_top1(item["model"], X[usable])
                    called[positions] = True
                    candidate[positions] = top
                    confidence[positions] = conf
                    accepted = conf >= threshold
                    missing_evidence = np.zeros(len(positions), dtype=bool)
                    if head == "make":
                        missing_evidence = ~candidate_make_supported([hosts[i] for i in positions],
                                                                    top, self.make_evidence)
                        accepted &= ~missing_evidence
                    labels[positions[accepted]] = top[accepted]
                    sources[positions[accepted]] = "B_expert"
                    reasons[positions] = np.where(accepted, "expert_threshold_pass",
                        np.where(missing_evidence, "no_manufacturer_evidence", "below_threshold"))
            frames.append(pd.DataFrame({
                "row_id": np.arange(n), "hostname": hosts, "head": head,
                "prediction": labels, "source": sources, "confidence": confidence,
                "threshold": threshold, "rule_pattern": patterns,
                "expert_called": called, "expert_candidate": candidate, "reason": reasons,
            }))
        return pd.concat(frames, ignore_index=True)

    def predict_row(self, hostname):
        trace = self.predict([hostname])
        return {row["head"]: {"top1": row["prediction"], "source": row["source"],
                             "confidence": float(row["confidence"]),
                             "is_unknown": row["prediction"] == HN_UNKNOWN,
                             "reason": row["reason"]}
                for _, row in trace.iterrows()}

hn_bundle = {
    "format": "sdc-hostname-cascade-v1",
    "unknown_label": HN_UNKNOWN,
    "heads": list(HN_HEADS),
    "rules": hn_rules,
    "vectorizer": hn_vectorizer,
    "experts": hn_experts,
    "normalization": "NFKC/lowercase/whitespace-v1",
}
hn_predictor = HostnameCascade(hn_bundle)
display(hn_predictor.predict([
    "desktop-abc123", "iphone", "galaxy-a51-cua-an", "A34-cua-Nga", "", None
])[["hostname", "head", "prediction", "source", "confidence", "reason"]])
# Dùng trên telemetry: hn_predictor.predict_row(row.get("dhcp_hostname"))


,hostname,head,prediction,source,confidence,reason
0,desktop-abc123,make,__unknown__,abstain,0.340348,no_manufacturer_evidence
1,iphone,make,Apple,A_rule,1.000000,rule_match
2,galaxy-a51-cua-an,make,Samsung,A_rule,1.000000,rule_match
3,a34-cua-nga,make,__unknown__,abstain,0.984358,no_manufacturer_evidence
4,,make,__unknown__,abstain,0.000000,invalid_hostname
5,,make,__unknown__,abstain,0.000000,invalid_hostname
6,desktop-abc123,type,Laptop,A_rule,1.000000,rule_match
7,iphone,type,Smartphone,A_rule,1.000000,rule_match
8,galaxy-a51-cua-an,type,Smartphone,A_rule,1.000000,rule_match
9,a34-cua-nga,type,Smartphone,B_expert,0.988588,expert_threshold_pass


In [6]:
# %pip install jinja2

# C5 — đo A, B và cascade trên cùng test; threshold giữ nguyên từ validation
def hn_score(frame, head, predictions, known_classes, system):
    truth = frame[head].fillna("__missing_truth__").astype(str).to_numpy()
    labeled = frame[head].notna().to_numpy()
    pred = pd.Series(predictions, dtype=object).fillna(HN_UNKNOWN).astype(str).to_numpy()
    answered = labeled & (pred != HN_UNKNOWN)
    correct = answered & (pred == truth)
    supported = labeled & np.isin(truth, list(known_classes))
    novel = labeled & ~np.isin(truth, list(known_classes))
    f1_labels = sorted(set(truth[supported]))
    return {
        "head": head, "system": system, "n_truth": int(labeled.sum()),
        "answered": int(answered.sum()),
        "selective_accuracy": float(correct.sum() / answered.sum()) if answered.any() else np.nan,
        "coverage": float(answered.sum() / labeled.sum()) if labeled.any() else np.nan,
        "overall_accuracy": float(correct.sum() / labeled.sum()) if labeled.any() else np.nan,
        "macro_f1_supported": float(f1_score(
            truth[supported], pred[supported], labels=f1_labels, average="macro", zero_division=0
        )) if supported.any() else np.nan,
        "novel_rows": int(novel.sum()),
        "novel_false_answer_rate": float((novel & answered).sum() / novel.sum()) if novel.any() else np.nan,
    }

hn_test_trace = hn_predictor.predict(hn_test["hostname"].tolist())
hn_X_test = hn_vectorizer.transform(hn_test["hostname"])
hn_test_nonzero = np.asarray(hn_X_test.getnnz(axis=1)).ravel() > 0
hn_scores = []
hn_source_scores = []
for hn_head in HN_HEADS:
    hn_item = hn_experts[hn_head]
    hn_known = set(hn_item["model"].classes_) | {y for _, h, y in hn_rules if h == hn_head}
    hn_a, _ = hn_rule_predictions(hn_test["hostname"].tolist(), hn_head, hn_rules)
    hn_top, hn_conf = hn_expert_top1(hn_item["model"], hn_X_test)
    hn_accept = hn_test_nonzero & (hn_conf >= hn_item["threshold"])
    if hn_head == "make":
        hn_accept &= candidate_make_supported(hn_test["hostname"].tolist(), hn_top,
                                               hn_predictor.make_evidence)
    hn_b = np.where(hn_accept, hn_top, HN_UNKNOWN)
    hn_c = hn_test_trace[hn_test_trace["head"].eq(hn_head)]["prediction"].to_numpy()
    for hn_name, hn_pred in [("A_rules", hn_a), ("B_expert", hn_b), ("C_cascade", hn_c)]:
        hn_scores.append(hn_score(hn_test, hn_head, hn_pred, hn_known, hn_name))
        for hn_source in ("AKIEN", "FIELD"):
            hn_source_mask = hn_test["source"].eq(hn_source).to_numpy()
            hn_source_scores.append({
                "source": hn_source,
                **hn_score(hn_test.loc[hn_source_mask], hn_head,
                           np.asarray(hn_pred)[hn_source_mask], hn_known, hn_name),
            })
hn_comparison = pd.DataFrame(hn_scores).set_index(["head", "system"]).sort_index()
display(hn_comparison.round(4))
hn_source_comparison = pd.DataFrame(hn_source_scores).set_index(["source", "head", "system"]).sort_index()
display(hn_source_comparison.round(4))
print("Metrics nguồn dùng source chính: hostname trùng AKIEN/FIELD được dedup và ghi về FIELD.")
print("Số FIELD test nhỏ: xem n_truth/answered; không coi tỷ lệ phần trăm là kết luận tổng thể.")
hn_attribution = pd.crosstab(hn_test_trace["head"], hn_test_trace["source"])
display(hn_attribution)
hn_gates = []
for hn_head in HN_HEADS:
    a = hn_comparison.loc[(hn_head, "A_rules")]
    c = hn_comparison.loc[(hn_head, "C_cascade")]
    checks = {
        "precision_ok": bool(c["selective_accuracy"] >= HN_TARGET_PRECISION - 0.02),
        "novel_fp_ok": bool(c["novel_false_answer_rate"] <= 0.10) if c["novel_rows"] else True,
        "overall_improved": bool(c["overall_accuracy"] > a["overall_accuracy"]),
    }
    hn_gates.append({"head": hn_head, **checks, "novel_tested": bool(c["novel_rows"]),
                     "PASS": all(checks.values())})
hn_gate = pd.DataFrame(hn_gates).set_index("head")
display(hn_gate)
print("Gate: novel_tested=False nghĩa là test không có class lạ để đo.")


n_truth  answered  selective_accuracy  coverage  \
head  system                                                       
make  A_rules      18495     11577              0.9947    0.6260   
      B_expert     18495      2594              0.9985    0.1403   
      C_cascade    18495     12068              0.9949    0.6525   
model A_rules      25300     18452              0.9795    0.7293   
      B_expert     25300     24993              0.9915    0.9879   
      C_cascade    25300     25112              0.9813    0.9926   
type  A_rules      55964     47327              0.9998    0.8457   
      B_expert     55964     55964              0.9996    1.0000   
      C_cascade    55964     55964              0.9997    1.0000   

                 overall_accuracy  macro_f1_supported  novel_rows  \
head  system                                                        
make  A_rules              0.6227              0.4136           5   
      B_expert             0.1400              0.3309           5   
      C_cascade            0.6492              0.5942           5   
model A_rules              0.7144              0.6412         145   
      B_expert             0.9794              0.7341         145   
      C_cascade            0.9740              0.8554         145   
type  A_rules              0.8455              0.5719           0   
      B_expert             0.9996              0.9895           0   
      C_cascade            0.9997              0.9929           0   

                 novel_false_answer_rate  
head  system                              
make  A_rules                     0.0000  
      B_expert                    0.0000  
      C_cascade                   0.0000  
model A_rules                     0.0207  
      B_expert                    0.4690  
      C_cascade                   0.4897  
type  A_rules                        NaN  
      B_expert                       NaN  
      C_cascade                      NaN

n_truth  answered  selective_accuracy  coverage  \
source head  system                                                       
AKIEN  make  A_rules      18495     11577              0.9947    0.6260   
             B_expert     18495      2594              0.9985    0.1403   
             C_cascade    18495     12068              0.9949    0.6525   
       model A_rules      25299     18452              0.9795    0.7294   
             B_expert     25299     24993              0.9915    0.9879   
             C_cascade    25299     25112              0.9813    0.9926   
       type  A_rules      55963     47327              0.9998    0.8457   
             B_expert     55963     55963              0.9996    1.0000   
             C_cascade    55963     55963              0.9997    1.0000   
FIELD  make  A_rules          0         0                 NaN       NaN   
             B_expert         0         0                 NaN       NaN   
             C_cascade        0         0                 NaN       NaN   
       model A_rules          1         0                 NaN    0.0000   
             B_expert         1         0                 NaN    0.0000   
             C_cascade        1         0                 NaN    0.0000   
       type  A_rules          1         0                 NaN    0.0000   
             B_expert         1         1              0.0000    1.0000   
             C_cascade        1         1              0.0000    1.0000   

                        overall_accuracy  macro_f1_supported  novel_rows  \
source head  system                                                        
AKIEN  make  A_rules              0.6227              0.4136           5   
             B_expert             0.1400              0.3309           5   
             C_cascade            0.6492              0.5942           5   
       model A_rules              0.7144              0.6412         144   
             B_expert             0.9795              0.7341         144   
             C_cascade            0.9741              0.8554         144   
       type  A_rules              0.8455              0.5719           0   
             B_expert             0.9996              0.9895           0   
             C_cascade            0.9997              0.9929           0   
FIELD  make  A_rules                 NaN                 NaN           0   
             B_expert                NaN                 NaN           0   
             C_cascade               NaN                 NaN           0   
       model A_rules              0.0000                 NaN           1   
             B_expert             0.0000                 NaN           1   
             C_cascade            0.0000                 NaN           1   
       type  A_rules              0.0000              0.0000           0   
             B_expert             0.0000              0.0000           0   
             C_cascade            0.0000              0.0000           0   

                        novel_false_answer_rate  
source head  system                              
AKIEN  make  A_rules                     0.0000  
             B_expert                    0.0000  
             C_cascade                   0.0000  
       model A_rules                     0.0208  
             B_expert                    0.4722  
             C_cascade                   0.4931  
       type  A_rules                        NaN  
             B_expert                       NaN  
             C_cascade                      NaN  
FIELD  make  A_rules                        NaN  
             B_expert                       NaN  
             C_cascade                      NaN  
       model A_rules                     0.0000  
             B_expert                    0.0000  
             C_cascade                   0.0000  
       type  A_rules                        NaN  
             B_expert                       NaN  
             C_cascade                      NaN

Metrics nguồn dùng source chính: hostname trùng AKIEN/FIELD được dedup và ghi về FIELD.
Số FIELD test nhỏ: xem n_truth/answered; không coi tỷ lệ phần trăm là kết luận tổng thể.


source,A_rule,B_expert,abstain
head,,,
make,23125,502,32337
model,30010,6946,19008
type,47327,8637,0


,precision_ok,novel_fp_ok,overall_improved,novel_tested,PASS
head,,,,,
make,True,True,True,True,True
type,True,True,True,False,True
model,True,False,True,True,False


Gate: novel_tested=False nghĩa là test không có class lạ để đo.


### C6. Lưu cùng run và thử lại

Các bảng bổ sung `hostname_source_comparison.csv`, `hostname_split_summary.csv`, `hostname_dataset_split.csv`, `hostname_field_examples.csv`, `hostname_conflicts.csv` và `hostname_class_support.csv` cho biết nguồn/split/nhãn xung đột và lớp nào thực sự được fit. Không dùng bảng FIELD train để báo cáo như test độc lập.

Artifact `hostname_cascade.joblib` chứa vectorizer, experts, thresholds và nguyên bộ luật A. Bảng `hostname_comparison.csv` đo A/B/cascade, `hostname_trace_test.csv` cho biết nguồn từng quyết định; `hostname_gate.csv` ghi head nào đạt tiêu chí thử nghiệm.

Nếu đang chạy trọn notebook 05, artifact vào cùng `run_dir`. Khi chỉ chạy phần C, tự tạo một run hostname riêng. Gate fail vẫn lưu để phân tích, kèm `experimental=True`. 09 nung cascade hostname vào sdc_iden.onnx; 12 ghép OS; 13 dựng cùng nhánh trong TFLite. Chạy riêng C sẽ sao chép iden/os của run nền mới nhất vào run C để pipeline có đủ ba bundle.


In [7]:
if HN_SAVE:
    hn_existing_run = globals().get("run_dir")
    if hn_existing_run is not None:
        hn_output_dir = Path(hn_existing_run)
    else:
        hn_run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f_hostname_" + HN_DATA_MODE)
        hn_output_dir = HN_ROOT / "Models" / hn_run_id / "train"
    # Chạy riêng C: sao chép rừng nền vào run mới để 09/12/13 dùng cùng run.
    import shutil
    hn_output_dir.mkdir(parents=True, exist_ok=True)
    if hn_existing_run is None:
        hn_base_runs = sorted(p for p in (HN_ROOT / "Models").iterdir()
                              if (p / "train/iden.joblib").is_file() and (p / "train/os.joblib").is_file())
        if not hn_base_runs:
            raise RuntimeError("Chạy phần A/B của 05 trước: chưa có run nền chứa iden.joblib và os.joblib")
        hn_base = hn_base_runs[-1]
        for name in ("iden.joblib", "os.joblib"):
            shutil.copy2(hn_base / "train" / name, hn_output_dir / name)
        print("Model gói tin/OS nền:", hn_base.name)
    hn_source_paths = [("AKIEN", "supervised_hostname", HN_CSV)]
    if HN_DATA_MODE == "combined":
        hn_source_paths += [("FIELD", "verified_session_join", HN_FIELD_SESSIONS),
                            ("FIELD", "verified_taxonomy", HN_FIELD_LABELS)]
        hn_source_paths += [("FIELD", "observed_dhcp_hostname", path) for path in HN_FIELD_DHCP]
    hn_provenance = {
        "format": "sdc-hostname-cascade-v1",
        "experimental": True,
        "created": datetime.now().isoformat(timespec="seconds"),
        "dataset": str(HN_CSV),
        "dataset_sha256": hashlib.sha256(HN_CSV.read_bytes()).hexdigest(),
        "data_mode": HN_DATA_MODE,
        "sources": [{"source": source, "role": role, "path": str(path),
                     "sha256": hashlib.sha256(Path(path).read_bytes()).hexdigest()}
                    for source, role, path in hn_source_paths],
        "field_load_audit": hn_field_load_audit,
        "dataset_audit": hn_data_audit,
        "field_sample_weight": HN_FIELD_WEIGHT,
        "rules_source": [str(HN_RULE_NOTEBOOK), str(HN_DEVICE_CATALOG)],
        "catalog_rules": {"catalog_sha256": hn_catalog["catalog_sha256"], "evidence": hn_catalog["evidence"],
                          "rules": len(hn_catalog["rules"]), "hierarchy": len(hn_catalog["hierarchy"])},
        "rules_sha256": hashlib.sha256(json.dumps(hn_rules, ensure_ascii=False).encode()).hexdigest(),
        "seed": HN_SEED, "test_size": 0.20, "validation_fraction_of_trainval": 0.20,
        "split": "connected_hostname_template_and_verified_FIELD_device", "min_train_per_class": HN_MIN_TRAIN,
        "max_features": HN_MAX_FEATURES, "target_precision": HN_TARGET_PRECISION,
        "threshold_selection": "validation per-head rule misses; fail closed if infeasible",
        "min_validation_answers": HN_MIN_VALID_ANSWERS,
        "split_rows": {k: len(v) for k, v in hn_splits.items()},
        "split_rows_by_source": {name: {str(source): int(count) for source, count in frame["source"].value_counts().items()}
                                 for name, frame in hn_splits.items()},
        "FIELD_fitted_rows_by_head": {
            h: int(hn_class_support.loc[hn_class_support["head"].eq(h)
                   & hn_class_support["supported"], "FIELD_train_hostnames"].sum()) for h in HN_HEADS},
        "thresholds": {h: float(hn_experts[h]["threshold"]) for h in HN_HEADS},
        "test_gate": {h: bool(hn_gate.loc[h, "PASS"]) for h in HN_HEADS},
        "note": "Hostname-only experiment; accuracy is not full packet pipeline accuracy.",
    }
    hn_hierarchy = {}
    for name, group in hn_train.dropna(subset=["model"]).groupby("model"):
        makes = group["make"].dropna().unique().tolist()
        types = group["type"].dropna().unique().tolist()
        hn_hierarchy[str(name)] = {"types": types}
        if len(makes) == 1:
            hn_hierarchy[str(name)]["make"] = makes[0]
    # Model chỉ có trong catalog (vd Galaxy S26) lấy make/type từ catalog; model đã học giữ hierarchy dữ liệu.
    for name, rule in hn_catalog["hierarchy"].items():
        hn_hierarchy.setdefault(name, rule)
    hn_bundle["hierarchy"] = hn_hierarchy
    hn_bundle["provenance"] = hn_provenance
    joblib.dump(hn_bundle, hn_output_dir / "hostname_cascade.joblib", compress=3)
    hn_comparison.reset_index().to_csv(hn_output_dir / "hostname_comparison.csv", index=False, encoding="utf-8-sig")
    hn_thresholds.reset_index().to_csv(hn_output_dir / "hostname_thresholds.csv", index=False, encoding="utf-8-sig")
    hn_gate.reset_index().to_csv(hn_output_dir / "hostname_gate.csv", index=False, encoding="utf-8-sig")
    hn_test_trace.to_csv(hn_output_dir / "hostname_trace_test.csv", index=False, encoding="utf-8-sig")
    hn_source_comparison.reset_index().to_csv(hn_output_dir / "hostname_source_comparison.csv", index=False, encoding="utf-8-sig")
    hn_split_summary.to_csv(hn_output_dir / "hostname_split_summary.csv", index=False, encoding="utf-8-sig")
    hn_split_manifest.to_csv(hn_output_dir / "hostname_dataset_split.csv", index=False, encoding="utf-8-sig")
    hn_field_manifest.to_csv(hn_output_dir / "hostname_field_examples.csv", index=False, encoding="utf-8-sig")
    hn_conflict_audit.to_csv(hn_output_dir / "hostname_conflicts.csv", index=False, encoding="utf-8-sig")
    hn_class_support.to_csv(hn_output_dir / "hostname_class_support.csv", index=False, encoding="utf-8-sig")
    # Lưu kèm catalog + rule đã sinh để run tự đủ, dựng lại được dù file chung đã sửa sau này.
    shutil.copy2(HN_DEVICE_CATALOG, hn_output_dir / "hostname_catalog.csv")
    (hn_output_dir / "hostname_catalog_rules.json").write_text(
        json.dumps(hn_catalog, ensure_ascii=False, indent=1), encoding="utf-8")
    hn_catalog_report.to_csv(hn_output_dir / "hostname_catalog_report.csv", index=False, encoding="utf-8-sig")
    (hn_output_dir / "hostname_meta.json").write_text(
        json.dumps(hn_provenance, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (hn_output_dir / "hostname_parity_inputs.json").write_text(
        json.dumps(hn_test["hostname"].sample(min(256, len(hn_test)), random_state=HN_SEED).tolist(), ensure_ascii=False),
        encoding="utf-8")
    print("RUN cho 09/12/13 =", repr(hn_output_dir.parent.name))
    if hn_existing_run is not None and isinstance(globals().get("meta"), dict):
        meta["hostname"] = hn_provenance
        write_meta()
    hn_reloaded = HostnameCascade.load(hn_output_dir / "hostname_cascade.joblib")
    hn_reload_probe = ["iphone", "galaxy-a51-cua-an", "desktop-abc123", None]
    pd.testing.assert_frame_equal(hn_predictor.predict(hn_reload_probe),
                                  hn_reloaded.predict(hn_reload_probe))
    print("Đã lưu và kiểm tra reload:", hn_output_dir / "hostname_cascade.joblib")
else:
    print("HN_SAVE=False: thử cascade trực tiếp qua hn_predictor, chưa lưu artifact.")


Model gói tin/OS nền: 20261014_094340


RUN cho 09/12/13 = '20261014_094625_087412_hostname_combined'


Đã lưu và kiểm tra reload: D:\01.AI_Security\02.SDC\Models\20261014_094625_087412_hostname_combined\train\hostname_cascade.joblib
